# Miniproyecto 4 - Reseñas a la carta

## GPT-2 condicionado sobre reseñas turísticas en español (Rest-Mex 2025)

Maestría · Universidad Icesi · Curso de Procesamiento de Lenguaje Natural

Autores: Juan José Aguado · Juan David Cruz · Juan Diego Ramírez

---

> **Sobre la reutilización del corpus.** Este notebook trabaja sobre el mismo corpus de los
> Miniproyectos 1 a 3. **Las Secciones 1 a 4.3 (entorno, corpus, EDA y protocolo) son las del
> Miniproyecto 1, sin modificar**: misma submuestra, mismo split y misma función de evaluación,
> de modo que lo que aquí se genera se puede medir contra las entregas anteriores. Lo nuevo
> empieza en §4.4: pasamos de clasificar reseñas a **generarlas**, siguiendo el notebook de la
> Sesión 4.

## 0. El problema

El notebook guía ajusta un GPT-2 en español sobre un corpus de chistes y concluye, mirando
algunas muestras, que el estilo cambia aunque los chistes no hagan reír. Es una evaluación a
ojo, y no puede ser otra cosa: los chistes no traen ninguna etiqueta contra la cual medir.

Nuestro corpus sí la trae. Cada reseña tiene sus estrellas y su tipo de establecimiento. Eso
cambia la tarea y permite medirla:

> **¿Puede un modelo generativo aprender no solo el estilo de las reseñas sino la polaridad
> que se le pide, y sirve lo que genera para algo más que leerlo?**

Ajustamos GPT-2 para que, dado un prefijo como `Reseña de hotel con 1 estrella:`, escriba una
reseña con esa polaridad. Y lo medimos de tres formas: **perplejidad** sobre reseñas reales que
nunca vio, **control** (un clasificador entrenado con reseñas reales, el modelo 1 de MP1, hace
de juez de lo generado) y **utilidad** (¿mejoran las reseñas sintéticas al clasificador que en
tres entregas no logró resolver 1★ y 2★?).

> **H1.** El *fine-tuning* baja la perplejidad muy por debajo del modelo base, y el prefijo de
> control la baja un poco más: saber la estrella ayuda a predecir el texto.
>
> **H2.** El juez reconoce la estrella pedida en los extremos (1★, 5★) y falla en 2★ a 4★, las
> mismas fronteras difusas que ningún clasificador de las entregas anteriores resolvió.
>
> **H3.** Añadir reseñas sintéticas de 1★ a 3★ al entrenamiento de TF-IDF + LogReg sube su
> macro-F1 más que duplicar las reseñas reales de esas clases.

### Mapa del notebook

| Sección | Contenido |
|---|---|
| 1 a 4.3 | Entorno, corpus, **EDA** y protocolo · *heredados del Miniproyecto 1, sin modificar* |
| 4.4 a 4.8 | Lo propio: configuración, tokenizador, formato con prefijo de control, jueces, métricas |
| 5 | El modelo base: tres GPT-2 en español, sin ajustar |
| 6 | **Técnica 1**: *fine-tuning* sin condición (como el guía) |
| 7 | **Técnica 2**: *fine-tuning* condicional por estrellas y tipo |
| 8 | Decodificación: el `generate` del guía corregido, y ocho estrategias comparadas |
| 9 | LoRA y otro checkpoint |
| 10 | El generador como clasificador |
| 11 | ¿Memoriza? |
| 12 | Aumento de datos para el clasificador de MP1 |
| 13 | Demo |
| 14 | Análisis cualitativo y modos de falla |
| 15 | Conclusiones y limitaciones |

---

> ### Nota sobre las Secciones 1 a 4
>
> Todo lo que sigue hasta los baselines de §4.3 está **copiado sin modificar** del notebook del
> Miniproyecto 1: mismo código, mismas gráficas, mismas lecturas. Garantiza que el test sobre el
> que se mide la perplejidad y el aumento de datos es exactamente el de las entregas anteriores.
>
> Sus textos hablan de los modelos de aquella entrega (TF-IDF, LSTM, BETO «excluido»...). No es
> un descuido: alterar el texto habría significado no reutilizar lo mismo. Al terminar el
> bloque, una celda puente retraduce cada hallazgo a las decisiones de este notebook. Algunas
> piezas heredadas (el Split B, las LSTM mencionadas) no se usan aquí; se conservan para no
> romper la identidad del bloque.

---

## 1. Entorno y reproducibilidad

Antes de cualquier análisis fijamos las condiciones que hacen que este notebook produzca los
mismos números en cada ejecución, y que corra igual en Colab con GPU, en Colab sin GPU y en
una máquina local.

<!-- REDACTAR: por qué la reproducibilidad no es un trámite sino parte del resultado:
     sin semilla fija, comparar cuatro modelos no significa nada. -->

In [ ]:
import sys, os, warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ejecutando en Colab: {IN_COLAB}')
print(f'Python: {sys.version.split()[0]}')

Instalamos únicamente lo que falte. En Colab casi todo viene preinstalado; el modelo de
vectores de spaCy en español es la descarga pesada (~570 MB) y solo se necesita a partir de
la sección 7.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

x = torch.randn(3, 3, device="cuda" if torch.cuda.is_available() else "cpu")
print("prueba tensor OK:", (x @ x).shape)

In [ ]:
# =========================================================================
# 1 · Entorno y reproducibilidad
# =========================================================================
import sys, os, platform, random, warnings, time
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- ¿Estamos en Colab? ---
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# --- Dependencias que podrían faltar en Colab: vectores de spaCy y ftfy ---
import spacy
if not spacy.util.is_package("es_core_news_lg"):
    print("Descargando es_core_news_lg (~570 MB, 1-2 min)...")
    os.system(f"{sys.executable} -m spacy download es_core_news_lg")
    print("Descarga terminada.")

try:
    import ftfy  # noqa: F401
except ImportError:
    os.system(f"{sys.executable} -m pip install -q ftfy")

### Semilla global

Fijamos `SEED = 42` en todas las fuentes de aleatoriedad del pipeline: submuestreo,
particiones, inicialización de pesos y orden de los lotes.

In [ ]:
# --- Semilla global ---
import numpy as np
import torch

SEED = 42

def fijar_semilla(sem: int = SEED):
    """Fija la semilla en random, numpy y torch (incl. CUDA)."""
    random.seed(sem)
    np.random.seed(sem)
    torch.manual_seed(sem)
    torch.cuda.manual_seed_all(sem)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla()
G = torch.Generator().manual_seed(SEED)   # barajado reproducible de los DataLoader

fijar_semilla()
print("\nSemilla global fijada en", SEED)

### Configuración adaptativa

El notebook debe correr en cualquier entorno, no fallar en el que no tenga GPU. Esta celda
detecta el dispositivo y ajusta tamaños de lote, número de épocas y estrategia para el
transformer. La configuración activa se imprime para que quede registrada junto a los
resultados.

In [ ]:
# --- Dispositivo y configuración adaptativa ---
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAY_GPU = DISPOSITIVO.type == "cuda"

if HAY_GPU:
    CFG = dict(submuestra=40_000, epocas=8, batch=128,
               emb_dim=128, hidden=128, max_vocab=30_000)
else:
    CFG = dict(submuestra=12_000, epocas=4, batch=64,
               emb_dim=96, hidden=96, max_vocab=25_000)
CFG.update(lr=2e-3, paciencia=3, seed=SEED)

# --- Traza del entorno (queda registrada en el .ipynb) ---
import sklearn, pandas as pd
try:
    import datasets
    ver_ds = datasets.__version__
except Exception:
    ver_ds = "no instalado"

print("Python      :", sys.version.split()[0], "| SO:", platform.system())
print("¿En Colab?  :", EN_COLAB)
print("Dispositivo :", DISPOSITIVO,
      f"({torch.cuda.get_device_name(0)})" if HAY_GPU else "")
print("Versiones   : torch", torch.__version__, "| sklearn", sklearn.__version__,
      "| pandas", pd.__version__, "| spacy", spacy.__version__, "| datasets", ver_ds)
print("\nCFG activa  :")
for k, v in CFG.items():
    print(f"  {k:11s}: {v}")

---

## 2. El corpus

Trabajamos con [`vg055/Rest-Mex2025`](https://huggingface.co/datasets/vg055/Rest-Mex2025):
208,051 reseñas turísticas en español sobre destinos de México, del shared task
**Rest-Mex 2025** de IberLEF. Licencia CC-BY-4.0.

**Por qué este corpus.** Rest-Mex 2025 además del texto y la polaridad (1–5), trae el pueblo, la región y el tipo de
establecimiento. Eso nos da tres cosas que otros corpus de sentimiento en español no ofrecen
a la vez: una **variable objetivo ordinal**, un **desbalance de clases real** (no fabricado)
y un **eje geográfico** con el que montar una prueba de generalización.

**Entrada de los modelos.** Concatenamos `Title` + `Review`. El título suele ser la señal
más condensada de la reseña («Todo excelente», «Nunca más»), y descartarlo sería desperdiciar información.

In [ ]:
# =========================================================================
# 2 · Carga del corpus Rest-Mex 2025
# =========================================================================
from datasets import load_dataset
from datetime import date

t0 = time.time()
ds = load_dataset("vg055/Rest-Mex2025", split="train")
df = ds.to_pandas()
print(f"Descarga + to_pandas: {time.time() - t0:.1f} s  |  fecha de descarga: {date.today().isoformat()}")

# --- Normalización mínima ---
df["Polarity"] = df["Polarity"].astype(int)
df["Title"]  = df["Title"].fillna("").astype(str).str.strip()
df["Review"] = df["Review"].fillna("").astype(str).str.strip()
df["texto"]  = (df["Title"] + ". " + df["Review"]).str.strip()   # entrada única de todos los modelos

# --- Inspección inicial ---
print("\nShape:", df.shape)
print("\nDtypes:")
print(df[["Title", "Review", "Polarity", "Town", "Region", "Type"]].dtypes.to_string())
display(df[["Title", "Review", "Polarity", "Town", "Region", "Type"]].head())

# --- Verificación contra docs/DATASET.md §3 (si no coincide, el corpus cambió en el Hub) ---
print("\n--- Verificación ---")
print(f"Registros          : {len(df):>8,}     (DATASET.md: 208,051)")
print(f"Regiones / Pueblos : {df['Region'].nunique():>3} / {df['Town'].nunique():<3}   (DATASET.md: 19 / 40)")
print(f"Tipos              : {sorted(df['Type'].unique())}")
print(f"Rango de Polarity  : {df['Polarity'].min()} - {df['Polarity'].max()}")
print("\nDistribución de Polarity (%):")
print((df["Polarity"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())
print("\nDistribución de Type (%):")
print((df["Type"].value_counts(normalize=True) * 100).round(2).to_string())
print("\nNulos por columna:")
print(df[["Title", "Review", "Polarity", "Town", "Region", "Type"]].isna().sum().to_string())

Antes de cualquier estadística, conviene ver el material crudo. Estas tres reseñas (una negativa, una intermedia y una positiva) dan la medida del registro, la extensión y el tipo
de lenguaje al que nos enfrentamos.

In [ ]:
# --- Tres reseñas crudas, una por nivel de polaridad ---
fijar_semilla()
for estrellas in (1, 3, 5):
    r = df[df["Polarity"] == estrellas].sample(1, random_state=SEED).iloc[0]
    print("=" * 100)
    print(f"[{estrellas}★]   Región: {r['Region']}   |   Pueblo: {r['Town']}   |   Tipo: {r['Type']}")
    print(f"Título : {r['Title']}")
    cuerpo = r["Review"]
    print(f"Reseña : {cuerpo[:800]}{'...' if len(cuerpo) > 800 else ''}")
    print(f"(longitud real del cuerpo: {len(cuerpo)} caracteres)\n")

Lo que se observa en el material crudo:

- **Registro coloquial y reseñas multi-tema.** Una misma reseña cubre reserva, transporte,
  trato y comida (1★) o comida, vino, ubicación y organización (5★). El modelo debe integrar
  señales dispersas, no clasificar una sola frase.
- **La reseña de 3★ es explícitamente mixta:** «gran vista» / «la comida no es buena» /
  «la cerveza es barata». Elogio y queja conviven. Anticipamos que 3★ será la clase más
  difícil, y aquí se ve por qué.
- **Ruido de codificación:** *CancÃºn* en vez de *Cancún*, mojibake por doble codificación
  UTF-8/Latin-1 en el CSV de origen.
- **Marcadores de truncamiento de TripAdvisor:** reseñas que terminan en «...Más» o «….Más»,
  restos del botón «leer más». Ruido a limpiar antes de tokenizar.
- **Texto a veces poco natural** («El barco marchamos antes, lo siento»): parte del corpus
  está traducido automáticamente o escrito por hablantes no nativos. Es dificultad inherente
  a los datos, no del modelo.

<!-- REDACTAR: qué se observa en los ejemplos. Registro coloquial, mezcla de temas dentro
     de una misma reseña, anglicismos, uso de mayúsculas y signos. Anticipar aquí que la
     reseña de 3 estrellas probablemente contenga elogios y quejas mezclados: eso explicará
     buena parte de los errores de la sección 14. -->

---

## 3. Análisis exploratorio

Esta sección no es un requisito administrativo: de aquí salen cuatro decisiones concretas
la métrica principal, la longitud máxima de secuencia, el diseño de la partición
geográfica y el tratamiento de la ordinalidad, y todas quedan justificadas con datos en
§3.8.

El análisis se hace sobre **el corpus completo** (208k). Solo el entrenamiento usará una
submuestra.

### 3.1 Calidad de los datos

Los nulos ya sabemos que son cero. Aquí buscamos lo que sí puede
contaminar el análisis y el entrenamiento: **duplicados exactos** (la misma reseña repetida
caería a la vez en train y test, es decir data-leakage), reseñas demasiado cortas para contener sentimiento, y dos defectos de origen que ya vimos en crudo: el **mojibake** de codificación y los **marcadores `...Más`** de TripAdvisor.

Diagnosticamos primero, decidimos después, y aplicamos solo la limpieza que se pueda
justificar.

In [ ]:
# =========================================================================
# 3.1 · Diagnóstico de calidad
# =========================================================================
import re

def tiene_mojibake(s: str) -> bool:
    return ("Ã" in s) or ("Â¿" in s) or ("Â¡" in s) or ("Âº" in s)

_pat_mas = re.compile(r"(?:\.{2,}|…)\s*Más\s*$")

rep = {
    "registros"                     : len(df),
    "Title vacío"                   : int((df["Title"] == "").sum()),
    "Review vacío"                  : int((df["Review"] == "").sum()),
    "Review < 15 caracteres"        : int((df["Review"].str.len() < 15).sum()),
    "duplicados exactos de 'texto'" : int(df["texto"].duplicated().sum()),
    "duplicados (texto, Polarity)"  : int(df.duplicated(["texto", "Polarity"]).sum()),
    "texto con mojibake"            : int(df["texto"].map(tiene_mojibake).sum()),
    "Review termina en '...Más'"    : int(df["Review"].str.contains(_pat_mas).sum()),
    "Polarity fuera de 1..5"        : int((~df["Polarity"].between(1, 5)).sum()),
}
display(pd.DataFrame({"chequeo": list(rep), "valor": list(rep.values())}))

print("\nEjemplos de mojibake (Title):")
for s in df.loc[df["texto"].map(tiene_mojibake), "Title"].head(5).tolist():
    print("  ", repr(s))

print("\nEjemplos de reseñas muy cortas:")
for s in df.loc[df["Review"].str.len() < 15, "Review"].head(5).tolist():
    print("  ", repr(s))

if rep["duplicados exactos de 'texto'"]:
    d = df[df["texto"].duplicated(keep=False)].sort_values("texto")
    print("\nEjemplo de par duplicado:")
    for _, r in d.head(2).iterrows():
        print(f"  [{r['Polarity']}★] {r['texto'][:130]}")

In [ ]:
# =========================================================================
# 3.1 · Limpieza (solo lo justificable) — corpus de trabajo para todo el notebook
# =========================================================================
import ftfy   # instalado en la Sección 1 si hacía falta

n0 = len(df)

# 1) Duplicados exactos: se elimina la copia (evita fuga train/test).
df = df.drop_duplicates("texto").reset_index(drop=True)

# 2) Mojibake + marcadores '...Más'. NO se eliminan filas: se corrige el texto.
def limpiar_texto(s: str) -> str:
    s = ftfy.fix_text(s)               # repara doble codificación UTF-8/Latin-1
    s = _pat_mas.sub("", s)            # quita el '...Más' de cola de TripAdvisor
    return s.strip()

df["Title"]  = df["Title"].map(limpiar_texto)
df["Review"] = df["Review"].map(limpiar_texto)
df["texto"]  = (df["Title"] + ". " + df["Review"]).str.strip()

# 3) Las reseñas cortas NO se eliminan: son parte del dominio y la §14 (errores) las usa.

print(f"Duplicados exactos eliminados : {n0 - len(df):,}")
print(f"Corpus de trabajo            : {len(df):,} reseñas")
print(f"texto con mojibake tras limpiar: {int(df['texto'].map(tiene_mojibake).sum())}")
print(f"Review termina en '...Más' tras limpiar: {int(df['Review'].str.contains(_pat_mas).sum())}")
print("\nAntes / después (fila 3 del head original):")
print("  ", repr(df['Title'].iloc[3]))

In [ ]:
# =========================================================================
# 3.1 · Chequeos rápidos finales
# =========================================================================
# a) ¿Cada pueblo pertenece a una única región? (si no, sería error de datos)
pueblos_multi = (df.groupby("Town")["Region"].nunique() > 1).sum()
print("Pueblos asignados a más de una región:", int(pueblos_multi))

# b) ¿Están todas las reseñas en español? Heurística por palabras funcionales sobre muestra.
_es = {" el ", " la ", " los ", " las ", " de ", " que ", " y ", " un ", " una ", " con ", " para "}
_en = {" the ", " and ", " is ", " was ", " we ", " very ", " of ", " to ", " this ", " they "}
def idioma_aprox(t):
    t = " " + t.lower() + " "
    return "en" if sum(w in t for w in _en) > sum(w in t for w in _es) else "es"
muestra = df["texto"].sample(5000, random_state=SEED)
prop_en = (muestra.map(idioma_aprox) == "en").mean()
print(f"Reseñas que parecen inglés (muestra de 5.000): {prop_en*100:.1f}%")

# c) Colas largas: ¿hay reseñas monstruosamente largas que distorsionen el EDA de longitudes?
p = np.percentile(df["texto"].str.len(), [50, 95, 99, 99.9, 100]).astype(int)
print(f"\nLongitud de 'texto' (caracteres) — P50 {p[0]} | P95 {p[1]} | P99 {p[2]} | P99.9 {p[3]} | máx {p[4]}")
print("Reseñas con > 4.000 caracteres:", int((df["texto"].str.len() > 4000).sum()))

# d) Title vacío: ¿molesta? (el 'texto' arranca con '. ' en esos casos)
print("Title vacío tras limpieza:", int((df["Title"] == "").sum()))

**Diagnóstico.** El corpus está en muy buen estado. Sobre 208.051 reseñas:

- **182 duplicados exactos** (0.09 %), todos con la misma polaridad, no hay etiquetas
  contradictorias. Se eliminan para que una misma reseña no caiga a la vez en train y test.
- **4.885 reseñas con mojibake** (2.3 %) por doble codificación UTF-8/Latin-1 en el CSV de
  origen. `ftfy` corrige el 99.8 %; quedan 8 casos residuales, irrelevantes.
- **16.248 reseñas** (7.8 %) terminaban en el marcador «...Más» de TripAdvisor. Se recorta
  esa cola; el resto del texto no se toca.
- **0 reseñas vacías o degeneradas**, **0 polaridades fuera de rango**, cada pueblo pertenece
  a una única región, y solo ~0.1 % de las reseñas está en otro idioma: el corpus es
  genuinamente español, como se anuncia.
- Longitudes muy asimétricas (mediana 284 caracteres, P99 1.424, máximo 8.299; 61 reseñas
  sobre 4.000).

**Decisiones de limpieza:**

| Problema | Decisión | Por qué |
|---|---|---|
| Duplicados exactos | eliminar (−182) | fuga de información train/test |
| Mojibake | corregir con `ftfy` | ruido de origen, no señal |
| Marcador «...Más» | recortar la cola | artefacto de scraping |
| Reseñas cortas | **conservar** | son parte del dominio|
| Reseñas muy largas | **conservar** | se acotan con `MAX_LEN`|
| Título vacío (6 casos) | **conservar** | despreciable; el cuerpo está intacto |
| Texto traducido / poco natural | **conservar** | dificultad real del corpus, no un defecto |

### 3.2 Distribución de la polaridad

La polaridad es la etiqueta que queremos predecir: las estrellas (1 a 5) que el autor le
puso a su reseña. Antes de entrenar nada miramos cómo se reparten esas estrellas, porque de
ese reparto dependen tanto la métrica con la que mediremos el éxito como la forma de entrenar.

In [ ]:
# =========================================================================
# 3.2 · Distribución de la polaridad
# =========================================================================
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
# Paleta fija por estrella, usada en TODO el notebook (rojo = malo, azul = bueno).
PALETA_POL = {1: "#b2182b", 2: "#ef8a62", 3: "#fddbc7", 4: "#67a9cf", 5: "#2166ac"}

conteo = df["Polarity"].value_counts().sort_index()
pct = conteo / conteo.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(conteo.index, conteo.values, color=[PALETA_POL[i] for i in conteo.index])
for b, n, p in zip(barras, conteo.values, pct):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{n:,}\n{p:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Polaridad (estrellas)")
ax.set_ylabel("Número de reseñas")
ax.set_title("Distribución de la polaridad en el corpus (207.869 reseñas)")
ax.set_ylim(0, conteo.max() * 1.15)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"n": conteo, "%": pct.round(2)}).to_string())
print(f"\nRatio mayoritaria/minoritaria : {conteo.max() / conteo.min():.1f} : 1")
print(f"Polaridad media / mediana     : {df['Polarity'].mean():.2f} / {df['Polarity'].median():.0f}")

In [ ]:
# =========================================================================
# 3.2 · Baseline de clase mayoritaria — de aquí sale la métrica principal
# =========================================================================
from sklearn.metrics import f1_score, accuracy_score

clase_mayoritaria = int(conteo.idxmax())
y_real = df["Polarity"].values
y_pred_may = np.full_like(y_real, clase_mayoritaria)

acc_may = accuracy_score(y_real, y_pred_may)
f1_may = f1_score(y_real, y_pred_may, average="macro", labels=[1, 2, 3, 4, 5], zero_division=0)
f1_por_clase = f1_score(y_real, y_pred_may, average=None, labels=[1, 2, 3, 4, 5], zero_division=0)

print(f"Baseline = predecir SIEMPRE {clase_mayoritaria}★ (sin leer el texto)")
print(f"  accuracy  = {acc_may:.4f}")
print(f"  macro-F1  = {f1_may:.4f}")
print(f"  F1 por clase 1..5 = {[round(v, 3) for v in f1_por_clase]}")

# Línea de referencia para TODAS las tablas de resultados.
BASELINE = {"clase": clase_mayoritaria, "accuracy": float(acc_may), "macro_f1": float(f1_may)}

La siguiente celda traduce ese desbalance en tres cosas concretas: cuánto pesará cada clase
al entrenar, si reformular la tarea ayudaría, y con cuántos ejemplos reales contaremos para
las clases poco frecuentes.

In [ ]:
# =========================================================================
# 3.2 (extra) · Qué implica el desbalance para el modelado
# =========================================================================
# a) Pesos de clase para la pérdida (inverso de la frecuencia, normalizado a media 1)
frec = conteo / conteo.sum()
w = (1 / frec) / (1 / frec).sum() * len(conteo)
print("class_weight (1..5):", {e: round(v, 2) for e, v in zip([1, 2, 3, 4, 5], w)})
print(f"  -> un error en 1★ pesará ~{w[1] / w[5]:.0f}x más que uno en 5★\n")

# b) ¿Se escapa del desbalance reformulando la tarea?
c3 = pd.Series(np.select([df["Polarity"] <= 2, df["Polarity"] == 3],
                         ["neg (1-2)", "neu (3)"], "pos (4-5)")).value_counts(normalize=True) * 100
c2 = pd.Series(np.where(df["Polarity"] >= 4, "pos (4-5)", "no-pos (1-3)")).value_counts(normalize=True) * 100
print("Colapsando a 3 clases (%):\n", c3.round(1).to_string())
print("\nColapsando a binario (%):\n", c2.round(1).to_string())

# c) Ejemplos de entrenamiento por clase tras la submuestra estratificada de 40k (split 80%)
proj = (frec * CFG["submuestra"] * 0.8).round().astype(int)
print(f"\nEjemplos de train por clase con submuestra de {CFG['submuestra']:,}:")
print({e: int(v) for e, v in zip([1, 2, 3, 4, 5], proj)})

# d) MAE del baseline mayoritario (para completar su fila en las tablas de resultados)
mae_may = float(np.abs(y_real - clase_mayoritaria).mean())
BASELINE["mae"] = mae_may
print(f"\nMAE del baseline 'siempre 5★': {mae_may:.3f} estrellas")

Dos de cada tres reseñas tienen 5 estrellas y solo cinco de cada cien tienen 1 o 2. Este sesgo hacia lo positivo es habitual en sitios de reseñas, así que no es un error a corregir sino una característica del problema.

La consecuencia práctica es que el accuracy (porcentaje de aciertos) deja de servir: un modelo que responda siempre "5 estrellas", sin leer el texto, acierta el 65,7 % de las
veces. Por eso medimos el éxito con el macro-F1, que evalúa la calidad clase por clase y luego promedia; ese mismo modelo perezoso saca un macro-F1 de 0,16, porque falla por
completo en cuatro de las cinco clases. Todo el notebook usa macro-F1 como criterio, y el accuracy solo se muestra al lado de este baseline como referencia.

Para que los modelos no ignoren las clases raras, al entrenar daremos a cada clase un peso inverso a su frecuencia, es decir, un error en una reseña de 1★ pesará unas 25 veces más que uno en
una de 5★.

Aun así el problema no desaparece: agrupando en positivo/neutro/negativo el corpus sigue siendo 87 % positivo, y al reducirlo a 40.000 reseñas para que todo corra en tiempo razonable quedarán apenas ~840 ejemplos de 1★ y ~845 de 2★ para entrenar. Es
suficiente para que el macro-F1 sea informativo, pero poco para conclusiones finas sobre esas dos clases; lo consideraremos en las limitaciones.

### 3.3 Tipo de establecimiento, región y pueblo

In [ ]:
# =========================================================================
# 3.3 · Tipo de establecimiento, región y pueblo
# =========================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

vc_type = df["Type"].value_counts()
axes[0].bar(vc_type.index, vc_type.values, color="#4c72b0")
for i, v in enumerate(vc_type.values):
    axes[0].text(i, v, f"{v / len(df) * 100:.0f}%", ha="center", va="bottom")
axes[0].set_title("Type (3 clases)"); axes[0].set_ylabel("reseñas")

vc_reg = df["Region"].value_counts()
sns.barplot(x=vc_reg.head(12).values, y=vc_reg.head(12).index, ax=axes[1], color="#4c72b0")
axes[1].set_title(f"Region — top 12 de {df['Region'].nunique()}"); axes[1].set_xlabel("reseñas")

vc_town = df["Town"].value_counts()
sns.barplot(x=vc_town.head(12).values, y=vc_town.head(12).index, ax=axes[2], color="#4c72b0")
axes[2].set_title(f"Town — top 12 de {df['Town'].nunique()}"); axes[2].set_xlabel("reseñas")

plt.tight_layout(); plt.show()

print("Type (%):")
print((vc_type / len(df) * 100).round(1).to_string())
print("\nConcentración geográfica:")
print(f"  Quintana Roo         : {vc_reg.iloc[0] / len(df) * 100:.1f}%")
print(f"  top 3 regiones       : {vc_reg.head(3).sum() / len(df) * 100:.1f}%")
print(f"  Tulum + Isla Mujeres : {vc_town.head(2).sum() / len(df) * 100:.1f}%")
print("\nRegiones completas (n y %):")
print(pd.DataFrame({"n": vc_reg, "%": (vc_reg / len(df) * 100).round(2)}).to_string())

El tipo de establecimiento está bastante repartido: 41.7 % restaurantes, 33.6 % atracciones (museos, playas, zonas arqueológicas) y 24.7 % hoteles. Es lo contrario al caso de la polaridad. Por eso, nos puede convenir usar `Type` como tarea de control: si un mismo modelo clasifica bien
el tipo pero mal las estrellas, la dificultad está en la tarea de polaridad, no en el modelo.

La geografía, en cambio, está muy concentrada. Quintana Roo (Cancún, Tulum, Isla Mujeres, Bacalar) reúne el 41 % de las reseñas, las tres regiones más grandes suman el 62 %, y solo
Tulum e Isla Mujeres ya son el 36 %. De las 19 regiones, muchas aportan menos del 1 %.

Aqui hay un riesgo para el modelo, si entrenamos y evaluamos mezclando al azar, el modelo ve durante el entrenamiento reseñas de los mismos hoteles y playas que luego tiene que calificar, y podría aprender a reconocer el destino (nombres propios, topónimos) en lugar del sentimiento.

In [ ]:
# (opcional) polaridad media por tipo y por región — anticipo de 3.6
print("Polaridad media por Type:")
print(df.groupby("Type")["Polarity"].mean().round(2).to_string())
print("\nPolaridad media — regiones del held-out del Split B vs. resto:")
ho = ["Chiapas", "Baja_CaliforniaSur", "Queretaro"]
print(f"  held-out : {df[df.Region.isin(ho)]['Polarity'].mean():.2f}")
print(f"  resto    : {df[~df.Region.isin(ho)]['Polarity'].mean():.2f}")

Los hoteles reciben calificaciones algo más bajas que atracciones y restaurantes (4,32
frente a 4,52 y 4,47), aunque la diferencia es pequeña y todas las medias siguen muy altas.
Y las tres regiones que podríamos dejar fuera para una prueba de generalización tienen una
polaridad media (4,44) casi idéntica a la del resto (4,45): esa eventual partición no
arrancaría sesgada por el lado de las etiquetas. Queda como observación.

<!-- LEER: el contraste entre un `Type` balanceado y una polaridad que no lo está, de ahí
     sale la tarea de control de la §13, y la concentración geográfica en Quintana Roo,
     que motiva la partición por región de la §10. -->

### 3.4 Longitud de las reseñas → de dónde sale `MAX_LEN`

Los notebooks guía fijan la longitud de secuencia en 256 o 512 tokens sin justificarlo.
Aquí la derivamos de la distribución real: `MAX_LEN` será el percentil 95 en tokens, y
reportaremos qué fracción de reseñas queda truncada.

In [ ]:
# =========================================================================
# 3.4 · Longitud de las reseñas (caracteres y tokens) -> MAX_LEN
# =========================================================================
import re

# Tokenizador de trabajo solo para medir longitudes. El definitivo se fija en 4.1,
# pero usa el mismo criterio: minúsculas, se parte por lo que no sea letra o número,
# se conservan tildes y ñ.
_re_tok = re.compile(r"[^a-záéíóúüñ0-9]+")
def _tok(t):
    return [x for x in _re_tok.sub(" ", t.lower()).split() if x]

df["n_car"] = df["texto"].str.len()

# Contar tokens en las 207k reseñas es lento; usamos una muestra grande y reproducible.
muestra_tok = df["texto"].sample(60000, random_state=SEED).map(lambda t: len(_tok(t)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].hist(df["n_car"].clip(upper=1500), bins=60, color="#4c72b0")
axes[0].set_title("Longitud en caracteres (recortada a 1500 para visualizar)")
axes[0].set_xlabel("caracteres"); axes[0].set_ylabel("reseñas")
axes[1].hist(muestra_tok.clip(upper=300), bins=60, color="#dd8452")
axes[1].set_title("Longitud en tokens (muestra de 60.000, recortada a 300)")
axes[1].set_xlabel("tokens"); axes[1].set_ylabel("reseñas")
plt.tight_layout(); plt.show()

pcts = [50, 75, 90, 95, 99]
tabla_long = pd.DataFrame({
    "percentil": pcts,
    "caracteres": np.percentile(df["n_car"], pcts).round().astype(int),
    "tokens": np.percentile(muestra_tok, pcts).round().astype(int),
})
print(tabla_long.to_string(index=False))

MAX_LEN = int(round(np.percentile(muestra_tok, 95) / 10) * 10)
trunc = (muestra_tok > MAX_LEN).mean()
print(f"\nMediana de tokens : {int(muestra_tok.median())}")
print(f"MAX_LEN (percentil 95 en tokens, redondeado a decena) = {MAX_LEN}")
print(f"Con ese límite se trunca el {trunc * 100:.1f}% de las reseñas (solo la cola larga).")

La mayoría de las reseñas son cortas: la mitad tiene 48 palabras o menos y el 90 % cabe en 136. Pero hay una cola larga que llega a más de 240. Fijamos el límite de longitud (`MAX_LEN`) en 150 palabras, que es el percentil 95: con eso solo se recorta el 4,4 % de las reseñas, y siempre por el final, que suele ser la parte menos informativa.

En los histogramas hay un pico alrededor de las 140 palabras (unos 800 caracteres) que parece un corte fijo del proceso de descarga de las reseñas, no algo propio del lenguaje. El pico del extremo derecho de cada gráfica es solo el efecto de amontonar ahí todo lo que se sale del rango dibujado; los valores reales están en la tabla de percentiles.

<!-- LEER: la asimetría de la distribución y por qué el P95 es el compromiso correcto. -->

### 3.5 Léxico distintivo por polaridad

Las frecuencias crudas están dominadas por palabras funcionales y no distinguen nada. Usamos
**log-odds ratio con prior de Dirichlet**, que mide qué tan sobrerrepresentado está un
término en una clase respecto al resto del corpus y es robusto ante términos poco frecuentes.

In [ ]:
# =========================================================================
# 3.5 · Léxico distintivo: 1★ frente a 5★ (log-odds con prior de Dirichlet)
# =========================================================================
from collections import Counter

def log_odds(textos_a, textos_b, alpha=0.01, top=18, min_freq=10):
    """z-score de log-odds (Monroe et al. 2008): qué términos distinguen A de B."""
    ca, cb = Counter(), Counter()
    for t in textos_a: ca.update(_tok(t))
    for t in textos_b: cb.update(_tok(t))
    vocab = set(ca) | set(cb)
    na, nb, a0 = sum(ca.values()), sum(cb.values()), alpha * len(vocab)
    z = {}
    for w in vocab:
        if ca[w] + cb[w] < min_freq:
            continue
        la = np.log((ca[w] + alpha) / (na + a0 - ca[w] - alpha))
        lb = np.log((cb[w] + alpha) / (nb + a0 - cb[w] - alpha))
        z[w] = (la - lb) / np.sqrt(1.0 / (ca[w] + alpha) + 1.0 / (cb[w] + alpha))
    ordenado = sorted(z.items(), key=lambda kv: kv[1])
    return ordenado[-top:][::-1], ordenado[:top]   # (más de A, más de B)

txt_1 = df.loc[df["Polarity"] == 1, "texto"]
txt_5 = df.loc[df["Polarity"] == 5, "texto"].sample(len(txt_1) * 2, random_state=SEED)
mas_1, mas_5 = log_odds(txt_1, txt_5)
LEXICO = {1: [w for w, _ in mas_1], 5: [w for w, _ in mas_5]}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, datos, titulo, color in [(axes[0], mas_1, "Más propio de 1★", "#b2182b"),
                                 (axes[1], mas_5, "Más propio de 5★", "#2166ac")]:
    palabras, scores = zip(*datos)
    ax.barh(range(len(palabras)), scores, color=color)
    ax.set_yticks(range(len(palabras))); ax.set_yticklabels(palabras); ax.invert_yaxis()
    ax.set_title(titulo); ax.set_xlabel("log-odds (z)")
plt.tight_layout(); plt.show()

print("LEXICO[1] (1★):", LEXICO[1])
print("LEXICO[5] (5★):", LEXICO[5])

En las reseñas de 1★ lo que más pesa es la negación: "no", "ni", "nada", "nunca". Es un hallazgo relevante en cuanto a elegir el modelo, porque un enfoque de bolsa de palabras (que cuenta términos sin mirar el orden) no puede distinguir "no recomiendo" de "recomiendo".
También aparecen mucho "dijeron", "dijo", "nos": en las reseñas malas la gente reconstruye conversaciones con el personal. Y "habitación" y "hotel" se inclinan al lado negativo, lo que encaja con que los hoteles tenían la media de estrellas más baja.

En 5★ dominan los adjetivos de entusiasmo ("excelente", "increíble", "hermoso", "amable", "agradable") mezclados con palabras de lugar ("playa", "ruinas", "lugar", "visita"). Esto
último conviene tenerlo en cuenta: parte de la señal no es sentimiento puro sino relacionado al tema (las atracciones reciben mejores calificaciones que los hoteles), así que un modelo podría acertar en parte fijándose en de qué habla la reseña y no en cómo la valora.

Guardamos ambas listas en `LEXICO` para tenerlas como referencia.

### 3.6 Interacciones

In [ ]:
# =========================================================================
# 3.6 · Interacciones: polaridad × tipo, × región, × longitud
# =========================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))

# 1) P(polaridad | tipo)
ct = pd.crosstab(df["Type"], df["Polarity"], normalize="index")
sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("P(polaridad | tipo)"); axes[0].set_xlabel("polaridad"); axes[0].set_ylabel("")

# 2) P(polaridad | región) para las 8 regiones con más reseñas
top8 = df["Region"].value_counts().head(8).index
cr = pd.crosstab(df.loc[df["Region"].isin(top8), "Region"], df["Polarity"], normalize="index")
cr = cr.loc[top8]  # ordenadas por volumen
sns.heatmap(cr, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("P(polaridad | región)"); axes[1].set_xlabel("polaridad"); axes[1].set_ylabel("")

# 3) longitud (caracteres) por polaridad
sns.boxplot(data=df, x="Polarity", y="n_car", hue="Polarity", legend=False,
            palette=PALETA_POL, showfliers=False, ax=axes[2])
axes[2].set_title("Longitud (caracteres) por polaridad")
axes[2].set_xlabel("polaridad"); axes[2].set_ylabel("caracteres")

plt.tight_layout(); plt.show()

print("Longitud mediana (caracteres) por polaridad:")
print(df.groupby("Polarity")["n_car"].median().to_string())
print("\nPolaridad media por región (top 8), de menor a mayor:")
print(df[df["Region"].isin(top8)].groupby("Region")["Polarity"].mean().round(2).sort_values().to_string())

El sesgo positivo es bastante parejo. Por tipo, los hoteles concentran algo más de reseñas negativas (4 % de 1★ y 4 % de 2★, frente al 1–2 % de las atracciones) y algo menos de 5★, aunque la diferencia es moderada.

Por región, la proporción de 5★ va del 57 % (Jalisco) al 70 % (Quintana Roo), y las tres regiones que podríamos apartar para una prueba de generalización (Chiapas, Baja California Sur, Querétaro) caen dentro de ese rango: apartarlas no cambiaría de forma brusca la mezcla de etiquetas.

La longitud sí se relaciona con la polaridad: las reseñas negativas son más largas (mediana de ~340–360 caracteres en 1★ y 2★, frente a ~275 en 5★).
Esto tiene sentido, quien se queja justifica el enojo con detalle, mientras que un "todo excelente" se despacha en una frase.

<!-- LEER: ¿son los hoteles más criticados que las atracciones? ¿escriben más largo quienes
     se quejan? Si la longitud correlaciona con la polaridad, es una señal que el modelo
     puede explotar y que conviene tener presente al interpretar resultados. -->

### 3.7 Cobertura del vocabulario frente a los embeddings preentrenados

El modelo 3 inicializará sus embeddings con vectores de spaCy entrenados sobre español
general. Antes de hacerlo, medimos cuánto de nuestro vocabulario turístico y regional está
efectivamente cubierto.

In [ ]:
# =========================================================================
# 3.7 · Cobertura del vocabulario en los vectores de spaCy (es_core_news_lg)
# =========================================================================
import spacy

nlp_vec = spacy.load("es_core_news_lg")
print("Vectores en es_core_news_lg:", nlp_vec.vocab.vectors.shape, "(nº de palabras × dimensiones)")

# Vocabulario del corpus (muestra de 50.000 reseñas para que sea rápido)
cont_vocab = Counter()
for t in df["texto"].sample(50000, random_state=SEED):
    cont_vocab.update(_tok(t))

tipos = list(cont_vocab)
con_vector = [w for w in tipos if nlp_vec.vocab[w].has_vector]
cob_tipos = len(con_vector) / len(tipos)
cob_tokens = sum(cont_vocab[w] for w in con_vector) / sum(cont_vocab.values())

oov = [(w, n) for w, n in cont_vocab.most_common() if not nlp_vec.vocab[w].has_vector][:25]

print(f"\nPalabras distintas en el corpus (muestra): {len(tipos):,}")
print(f"Cobertura por TIPOS  (palabras distintas con vector): {cob_tipos * 100:.1f}%")
print(f"Cobertura por TOKENS (apariciones cubiertas)        : {cob_tokens * 100:.1f}%")
print("\nPalabras frecuentes SIN vector (OOV):")
for w, n in oov:
    print(f"  {w:22s} {n}")

Casi todas las palabras que aparecen en las reseñas (99,1 % de las apariciones) tienen vector preentrenado. La cobertura baja al 74 % si contamos palabras distintas, pero lo que
falta son casi en su totalidad nombres propios: destinos (Bacalar, Sayulita, Taxco, Chichén,
Teotihuacán) y nombres de hoteles y restaurantes (Ziggy, Rolandi, Nautibeach). Vocabulario
de sentimiento sin cubrir prácticamente no hay.

Esto tiene dos lecturas. Partir de vectores preentrenados es una buena apuesta, porque el modelo no tiene que aprender desde cero qué significan "excelente" o "pésimo". Y como los nombres de lugar no tienen vector, un modelo que los use tal cual no puede apoyarse en el destino para adivinar la nota; en cambio, si dejamos que ajuste los vectores durante el entrenamiento, podría construir representaciones para esos nombres y usarlas como atajo.

Por eso conviene probar las dos variantes, vectores congelados y vectores ajustados, y comparar.
También se nota algo de ruido ortográfico: "pátzcuaro" y "patzcuaro", "teotihuacán" y "teotihuacan", cuentan como palabras distintas por la tilde.

In [ ]:
# =========================================================================
# 3.7b · Tres cifras para cerrar el diagnóstico
# =========================================================================
# (1) ¿Por qué un vocabulario de 30.000? Cobertura de apariciones según el corte.
cont_full = Counter()
for t in df["texto"].sample(80000, random_state=SEED):
    cont_full.update(_tok(t))
total = sum(cont_full.values())
print("Cobertura de apariciones según el tamaño de vocabulario:")
for k in (10000, 20000, 30000, 50000):
    cub = sum(n for _, n in cont_full.most_common(k))
    print(f"  vocab {k:>6,} -> {cub / total * 100:.2f}%")
print(f"  (palabras distintas en la muestra: {len(cont_full):,})")

# (2) Ruido de etiqueta: texto que contradice las estrellas.
neg_fuerte = {"pésimo", "pesimo", "horrible", "terrible", "asqueroso", "sucio", "estafa",
              "engaño", "robo", "nunca", "jamás", "malísimo", "peor"}
pos_fuerte = {"excelente", "maravilloso", "hermoso", "increíble", "perfecto", "espectacular",
              "encantó", "encanto", "recomiendo"}
def contiene(texto, palabras):
    return len(set(_tok(texto)) & palabras) > 0

r5 = df.loc[df["Polarity"] == 5, "texto"].map(lambda t: contiene(t, neg_fuerte)).mean()
r1 = df.loc[df["Polarity"] == 1, "texto"].map(lambda t: contiene(t, pos_fuerte)).mean()
print(f"\nReseñas de 5★ con vocabulario muy negativo en el texto: {r5 * 100:.1f}%")
print(f"Reseñas de 1★ con vocabulario muy positivo en el texto: {r1 * 100:.1f}%")

# (3) Densidad de negación por polaridad (negaciones por cada 100 palabras).
neg_tok = {"no", "ni", "nada", "nunca", "tampoco", "jamás", "sin"}
def dens_neg(texto):
    toks = _tok(texto)
    return sum(w in neg_tok for w in toks) / max(len(toks), 1) * 100
print("\nNegaciones por cada 100 palabras, según la polaridad:")
for p in (1, 2, 3, 4, 5):
    s = df.loc[df["Polarity"] == p, "texto"].sample(
        min(5000, int((df["Polarity"] == p).sum())), random_state=SEED)
    print(f"  {p}★: {s.map(dens_neg).mean():.2f}")

### 3.8 Síntesis: del hallazgo a la decisión

<!-- REDACTAR - la celda más importante del EDA. Una lista numerada, cada línea con la
     forma «hallazgo → decisión → dónde se aplica»:

     1. Desbalance 65/22/7/3/3  →  macro-F1 como métrica principal  →  §4, todas las tablas
     2. Distribución de longitudes  →  MAX_LEN = P95  →  §4
     3. Concentración en Quintana Roo  →  partición geográfica  →  §10
     4. Objetivo ordinal  →  MAE y QWK + formulaciones ordinales  →  §9
     5. Cobertura de embeddings  →  expectativa sobre el modelo 3  →  §7
     6. `Type` balanceado  →  tarea de control  →  §13
-->

| Hallazgo del EDA | Decisión que induce |
|---|---|
| El corpus trae mojibake y colas "...Más" de TripAdvisor, pero cero reseñas vacías y solo 182 duplicados exactos. | Se corrige el texto (`ftfy` + recorte) y se eliminan los duplicados; nada más. Corpus de trabajo: 207.869 reseñas. |
| El 66 % de las reseñas son 5★; predecir siempre "5" da 65,7 % de accuracy pero 0,16 de macro-F1. | La métrica principal es **macro-F1**. El accuracy solo se reporta junto a este baseline. |
| Las clases 1★ y 2★ juntas son el 5 %; tras reducir a 40.000 reseñas quedan ~840 ejemplos de cada una. | Se pondera cada clase por el inverso de su frecuencia al entrenar, y se asume desde ya que las métricas de 1★ y 2★ serán ruidosas (limitación). |
| Longitudes muy asimétricas: mediana 48 palabras, percentil 95 = 148. | **`MAX_LEN` = 150 palabras** (percentil 95); trunca solo el 4 % de las reseñas. |
| En 1★ domina la negación: 3,75 por cada 100 palabras, frente a 1,11 en 5★. | Una bolsa de palabras no distingue "no recomiendo" de "recomiendo": se justifica probar modelos que miran el orden (LSTM) y en ambos sentidos (bidireccional). |
| El 99 % de las apariciones tiene vector preentrenado; lo que falta son nombres de destinos y de hoteles. | Se inicializan los embeddings con los de spaCy y se prueban **congelados y ajustados**, porque ajustar podría reintroducir el destino como atajo. |
| Quintana Roo es el 41 % del corpus; con partición aleatoria el modelo ve en entrenamiento los mismos destinos que evalúa. | Se prueba además una **partición geográfica** (Chiapas, Baja California Sur y Querétaro fuera del entrenamiento) para medir generalización a destinos nuevos. |
| La escala 1–5 está ordenada: confundir 4★ con 5★ no equivale a confundir 1★ con 5★. | Además de macro-F1 se reportan **MAE y QWK**, y se prueba una formulación de regresión con umbrales. |
| `Type` está equilibrado (42/34/25) y no es ordinal. | Sirve de **tarea de control**: la misma arquitectura sobre `Type` dirá si un macro-F1 flojo en polaridad es culpa de la tarea o del modelo. |
| El 15 % de las reseñas de 1★ usan vocabulario muy positivo (y el 3 % de las de 5★, muy negativo). | Hay ruido de etiqueta y reseñas mixtas; ponen un techo al rendimiento y se analizan en la sección de errores. |

---

## 4. Preprocesamiento y protocolo experimental

Para que la comparación entre cuatro modelos signifique algo, todos deben ver exactamente
los mismos datos, la misma partición y las mismas métricas. Esta sección fija ese contrato.

### 4.1 Tokenización

Partimos del tokenizador del notebook guía y corregimos dos cosas: construimos el vocabulario
**por frecuencia** (el notebook 4 lo hace por orden de aparición, ver `docs/DECISIONS.md`
§D-002) y verificamos que la normalización preserve tildes y ñ.

In [ ]:
# =========================================================================
# 4.1 · Tokenizador definitivo y construcción del vocabulario
# =========================================================================
_RE_TOKEN = re.compile(r"[^a-záéíóúüñ0-9]+")

def tokenizar(texto: str):
    """Minúsculas; se parte por todo lo que no sea letra o número; se conservan
    tildes y ñ. Mismo criterio que se usó en el EDA."""
    return [t for t in _RE_TOKEN.sub(" ", texto.lower()).split() if t]

PAD, UNK = 0, 1

def construir_vocab(textos, max_vocab):
    """Vocabulario por FRECUENCIA (most_common), con [PAD]=0 y [UNK]=1 reservados.
    El notebook 4 del curso lo construye por orden de aparición: sobre un Counter
    eso no da los tokens más frecuentes sino los que aparecieron primero."""
    cont = Counter()
    for t in textos:
        cont.update(tokenizar(t))
    vocab = {"[PAD]": PAD, "[UNK]": UNK}
    for palabra, _ in cont.most_common(max_vocab - 2):
        vocab[palabra] = len(vocab)
    return vocab

ej = df["texto"].iloc[0]
print("Reseña :", ej[:150], "...")
print("Tokens :", tokenizar(ej)[:25], "...")
print("\nEl vocabulario definitivo se construye en 4.2, solo con el texto de entrenamiento.")

In [ ]:
# --- Cómo se convierte una reseña en entrada para el modelo: tokens -> ids -> padding ---
def codificar(texto, vocab, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK) for t in tokenizar(texto)[:max_len]]
    n_real = max(len(ids), 1)                 # longitud antes del relleno
    ids += [PAD] * (max_len - len(ids))
    return ids, n_real

# Vocabulario temporal (solo para esta demostración; el real se hace en 4.2)
_vocab_demo = construir_vocab(df["texto"], CFG["max_vocab"])
_id2tok = {i: t for t, i in _vocab_demo.items()}

frase = "El hotel estaba sucio y el sargazo cubría toda la playa. No volvería."
ids, n = codificar(frase, _vocab_demo, max_len=20)
print("Texto        :", frase)
print("Tokens       :", tokenizar(frase))
print("IDs          :", ids)
print("Reconstruido :", [_id2tok[i] for i in ids])
print(f"\n{n} tokens reales; los otros {20 - n} son [PAD] (relleno para que todas las")
print("secuencias tengan el mismo largo y se puedan procesar por lotes).")
del _vocab_demo, _id2tok

El tokenizador parte el texto en palabras, pasa todo a minúsculas y elimina signos y saltos
de línea, pero conserva tildes y ñ (importante en español). Cada palabra se sustituye por un
número, que es su posición en el vocabulario; las palabras que no estén en el vocabulario se
marcan como `[UNK]`, y las secuencias más cortas que `MAX_LEN` se completan con `[PAD]` para
que todos los ejemplos tengan el mismo largo y se puedan procesar por lotes.

El vocabulario se ordena por frecuencia: las palabras más comunes ("el", "y", "la") reciben
los números más bajos y las raras ("cubría"), los más altos. En el ejemplo, las 13 palabras
reales caben de sobra en el límite de 150 y ninguna quedó fuera del vocabulario.

<!-- LEER: tamaño del vocabulario, tasa de UNK, qué se pierde al truncar. -->

### 4.2 Submuestra y particiones

Entrenamos sobre 40,000 reseñas estratificadas por polaridad. **No balanceamos las clases**:
el desbalance es una propiedad del dominio que queremos estudiar, no un defecto que ocultar.

In [ ]:
# =========================================================================
# 4.2 · Submuestra estratificada
# =========================================================================
from sklearn.model_selection import train_test_split

N_SUB = min(CFG["submuestra"], len(df))
sub, _ = train_test_split(df, train_size=N_SUB, random_state=SEED, stratify=df["Polarity"])
sub = sub.reset_index(drop=True)

comp = pd.DataFrame({
    "corpus (%)":     (df["Polarity"].value_counts(normalize=True).sort_index() * 100).round(2),
    "submuestra (%)": (sub["Polarity"].value_counts(normalize=True).sort_index() * 100).round(2),
    "n en submuestra": sub["Polarity"].value_counts().sort_index(),
})
print(f"Submuestra: {len(sub):,} reseñas (de {len(df):,}), estratificada por polaridad.")
print(comp.to_string())

La submuestra de 40.000 reseñas conserva exactamente las proporciones del corpus (66 % de
5★, 2,6 % de 1★).

In [ ]:
# =========================================================================
# 4.2 · Split A — aleatorio estratificado 80/10/10 (protocolo estándar)
# =========================================================================
idx = np.arange(len(sub))
y = sub["Polarity"].values

idx_tr, idx_tmp = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=y)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=SEED, stratify=y[idx_tmp])
SPLIT_A = {"train": idx_tr, "val": idx_va, "test": idx_te}

print("Split A:")
for k, v in SPLIT_A.items():
    d = (pd.Series(y[v]).value_counts(normalize=True).sort_index() * 100).round(1)
    print(f"  {k:5s}: {len(v):>6,} reseñas  |  % por estrella: {d.tolist()}")

# Vocabulario definitivo: SOLO con el texto de entrenamiento de Split A (sin fuga)
vocab = construir_vocab(sub.iloc[SPLIT_A["train"]]["texto"], CFG["max_vocab"])
id2tok = {i: t for t, i in vocab.items()}
print(f"\nVocabulario definitivo: {len(vocab):,} tokens")

# Tasa de [UNK] y de truncamiento sobre el conjunto de test
unk = tot = trunc = 0
for t in sub.iloc[SPLIT_A["test"]]["texto"]:
    toks = tokenizar(t)
    tot += len(toks); unk += sum(w not in vocab for w in toks); trunc += len(toks) > MAX_LEN
print(f"En test -> tokens fuera del vocabulario: {unk / tot * 100:.1f}%  |  "
      f"reseñas truncadas por MAX_LEN: {trunc / len(SPLIT_A['test']) * 100:.1f}%")

El Split A la divide en 80/10/10 manteniendo esas proporciones en las tres partes, así que entrenamiento, validación y prueba son comparables entre sí. El vocabulario definitivo (30.000 palabras, construido solo con el texto de entrenamiento) deja fuera apenas el 1,5 % de las palabras del test, y el límite de 150 palabras trunca el 4,2 %, es decir, perdidas pequeñas.

In [ ]:
# =========================================================================
# 4.2 · Split B — geográfico (regiones fuera del entrenamiento)
# =========================================================================
REGIONES_FUERA = ["Chiapas", "Baja_CaliforniaSur", "Queretaro"]

es_fuera = sub["Region"].isin(REGIONES_FUERA).values
idx_b_test = idx[es_fuera]
idx_b_rest = idx[~es_fuera]
idx_b_tr, idx_b_va = train_test_split(idx_b_rest, test_size=0.12, random_state=SEED,
                                      stratify=y[idx_b_rest])
SPLIT_B = {"train": idx_b_tr, "val": idx_b_va, "test": idx_b_test}

print(f"Regiones fuera del entrenamiento: {REGIONES_FUERA}")
for k, v in SPLIT_B.items():
    print(f"  {k:5s}: {len(v):>6,} reseñas")

print("\nComprobación de que el test geográfico no está sesgado:")
print(f"  polaridad media : {sub.iloc[idx_b_test]['Polarity'].mean():.2f}  vs  "
      f"{sub.iloc[idx_b_rest]['Polarity'].mean():.2f}  (resto)")
print(f"  % hoteles       : {(sub.iloc[idx_b_test]['Type'] == 'Hotel').mean() * 100:.0f}%  vs  "
      f"{(sub.iloc[idx_b_rest]['Type'] == 'Hotel').mean() * 100:.0f}%  (resto)")
print(f"  % 5★            : {(sub.iloc[idx_b_test]['Polarity'] == 5).mean() * 100:.0f}%  vs  "
      f"{(sub.iloc[idx_b_rest]['Polarity'] == 5).mean() * 100:.0f}%  (resto)")

El Split B aparta tres regiones enteras (Chiapas, Baja California Sur y Querétaro; 7.456 reseñas, el 18,5 %). La polaridad media se ve casi igual (4,43 vs 4,46) y la proporción de 5★ es casi igual (64 % vs 66 %). Tiene algo más de hoteles (28 % vs 24 %), que suelen recibir peores notas, así que si el rendimiento cae en el Split B habrá que descontar esa pequeña diferencia.

### 4.3 Métricas y baselines

Una única función de evaluación que devuelve las seis métricas, usada sin excepción por
todos los modelos del notebook. Es lo que hace que la tabla final sea comparable.

In [ ]:
# =========================================================================
# 4.3 · Función de evaluación única — todos los modelos la usan sin excepción
# =========================================================================
from sklearn.metrics import (f1_score, accuracy_score, mean_absolute_error,
                             cohen_kappa_score, confusion_matrix, classification_report)

ESTRELLAS = [1, 2, 3, 4, 5]
RESULTADOS = []   # una fila por modelo
PREDS = {}        # nombre -> (y_true, y_pred) en el test de Split A, para el análisis de errores

def evaluar(y_true, y_pred, nombre, registrar=True, n_params=None, segundos=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    f1c = f1_score(y_true, y_pred, average=None, labels=ESTRELLAS, zero_division=0)
    fila = {
        "modelo":   nombre,
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=ESTRELLAS, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "mae":      mean_absolute_error(y_true, y_pred),
        "qwk":      cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=ESTRELLAS),
        **{f"f1_{e}": v for e, v in zip(ESTRELLAS, f1c)},
        "n_params": n_params, "segundos": segundos,
    }
    if registrar:
        RESULTADOS.append(fila)
    print(f"{nombre:38s} | macro-F1 {fila['macro_f1']:.4f} | acc {fila['accuracy']:.4f} "
          f"| MAE {fila['mae']:.3f} | QWK {fila['qwk']:.4f}")
    return fila

print("evaluar() lista. Métricas que devuelve:")
print("  macro-F1 : media de la F1 de las 5 clases (métrica principal del notebook)")
print("  accuracy : % de aciertos (solo se lee junto al baseline mayoritario)")
print("  MAE      : error medio en estrellas (0.5 = se equivoca media estrella de media)")
print("  QWK      : acuerdo con la nota real penalizando más los errores grandes (1=perfecto, 0=azar)")
print("  f1_1..f1_5 : F1 de cada clase por separado")

In [ ]:
# =========================================================================
# 4.3 · Baselines — ningún modelo es "útil" si no los supera en macro-F1
# =========================================================================
y_tr_A = sub.iloc[SPLIT_A["train"]]["Polarity"].values
y_te_A = sub.iloc[SPLIT_A["test"]]["Polarity"].values

# 1) Clase mayoritaria: predecir siempre la estrella más frecuente del entrenamiento
clase = int(pd.Series(y_tr_A).value_counts().idxmax())
evaluar(y_te_A, np.full_like(y_te_A, clase), f"Baseline · siempre {clase}★")

# 2) Azar estratificado: sortear según la distribución del entrenamiento
fijar_semilla()
p = pd.Series(y_tr_A).value_counts(normalize=True).sort_index()
y_azar = np.random.choice(p.index.values, size=len(y_te_A), p=p.values)
evaluar(y_te_A, y_azar, "Baseline · azar estratificado")

pd.DataFrame(RESULTADOS)[["modelo", "macro_f1", "accuracy", "mae", "qwk"]].round(4)

Los dos baselines dejan claro por qué hacen falta varias métricas a la vez. Predecir siempre "5★" tiene buen accuracy (0,66) pero macro-F1 de 0,16, porque cuatro clases quedan en cero.
Sortear al azar respetando las proporciones sube el macro-F1 a 0 19 (todas las clases reciben algo de acierto) pero hunde el accuracy a 0,48 y empeora el error medio a 0,83 estrellas. Y el QWK de ambos es prácticamente cero: ninguno tiene el más mínimo acuerdo real con la nota que puso el autor.

Un modelo solo cuenta como útil si supera el macro-F1 del azar (0,19) sin caer por debajo del accuracy de la clase mayoritaria (0,66), y si además consigue un QWK claramente positivo.

---

> ### Fin del bloque heredado: del EDA a esta entrega
>
> Aquí termina lo copiado del Miniproyecto 1. La tabla de §3.8 mapea hallazgos a aquel
> notebook; esta los retraduce a un problema de generación.

| Hallazgo del EDA | Qué decide en esta entrega | Dónde |
|---|---|---|
| 66 % de 5★ y solo ~2,6 % de 1★ y de 2★ | Un generador sin condición escribirá sobre todo reseñas positivas: el control por prefijo es necesario. Y las clases escasas son las candidatas al aumento de datos | §7, §12 |
| Mediana 48 palabras, P95 = 148 | `MAX_LEN_GPT` = P95 **en tokens BPE, con el prefijo**; `max_new_tokens` de generación según la longitud típica | §4.5, §8 |
| Registro coloquial, mexicanismos, nombres de destinos | El preentrenamiento importa: comparamos tres GPT-2 en español entrenados sobre corpus distintos | §5 |
| En 1★ domina la negación | Se revisa si las reseñas generadas a 1★ la reproducen | §14 |
| `Type` balanceado, con léxico propio | Segundo control (tipo de establecimiento), con su propio juez | §4.7, §7 |

In [ ]:
# --- Verificación: el bloque heredado es idéntico al del Miniproyecto 1 ---
# Se ejecuta solo si el notebook de MP1 está disponible (en local); en Colab se omite.
import json, pathlib
_ruta_mp1 = pathlib.Path("../../miniproyecto 1/notebooks/miniproyecto1_restmex.ipynb")
if _ruta_mp1.exists():
    _mp1 = json.loads(_ruta_mp1.read_text(encoding="utf8"))["cells"][3:71]
    _nb = json.loads(pathlib.Path("miniproyecto4_restmex_gpt.ipynb").read_text(encoding="utf8"))["cells"]
    _her = [c for c in _nb if "heredado-mp1" in c.get("metadata", {}).get("tags", [])]
    _txt = lambda c: "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
    assert len(_her) == len(_mp1) and all(_txt(a) == _txt(b) for a, b in zip(_her, _mp1))
    print(f"OK: las {len(_her)} celdas heredadas son idénticas a las del Miniproyecto 1.")
else:
    print("Notebook de MP1 no disponible en este entorno; verificación omitida.")

Comprobamos que el protocolo heredado se reprodujo: si los baselines no dan los números de MP1,
el test no es el mismo y ninguna comparación con las entregas anteriores sería válida.

In [ ]:
# --- Bloqueante: los baselines heredados reproducen los del Miniproyecto 1 ---
if CFG["submuestra"] == 40_000:
    assert abs(RESULTADOS[0]["accuracy"] - 0.6565) < 1e-3, "El split no reproduce el de MP1"
    assert abs(RESULTADOS[0]["macro_f1"] - 0.1585) < 1e-3, "El split no reproduce el de MP1"
    print("OK: submuestra y Split A idénticos a los del Miniproyecto 1.")
else:
    print("Configuración degradada (sin GPU): la comparación con MP1 no aplica.")

### 4.4 Dependencias y configuración propias

La configuración heredada (`CFG`) es la de MP1 y no se toca. El generador tiene la suya:
`CFG_GPT` para las corridas principales y `CFG_EST`, reducida, para los estudios de §9. En CPU
el notebook no falla: limita el tamaño del entrenamiento y el número de muestras, y lo anuncia.

In [ ]:
# =========================================================================
# 4.4 · Dependencias y configuración del generador
# =========================================================================
def _asegurar(modulo, paquete=None):
    try:
        __import__(modulo)
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {paquete or modulo}")

for m in ["transformers", "accelerate", "peft"]:
    _asegurar(m)

import contextlib, io, tempfile, shutil, math
import transformers
from transformers import set_seed
from pathlib import Path

CHECKPOINT = "mrm8488/spanish-gpt2"          # el del notebook guía
OTROS_CHECKPOINTS = {"DeepESP (libros)": "DeepESP/gpt2-spanish",
                     "datificate (Wikipedia)": "datificate/gpt2-small-spanish"}

if HAY_GPU:
    CFG_GPT = dict(epocas=2, batch=16, batch_eval=32, max_len_tope=160, n_train_max=None,
                   n_eval=1_000, n_dec=20, n_aug=1_000, max_new=90, batch_gen=64)
    CFG_EST = dict(n_train=8_000, epocas=1)
else:
    print("⚠ Sin GPU: configuración degradada (resultados NO comparables con la corrida de referencia).")
    CFG_GPT = dict(epocas=1, batch=8, batch_eval=16, max_len_tope=96, n_train_max=800,
                   n_eval=120, n_dec=2, n_aug=30, max_new=40, batch_gen=16)
    CFG_EST = dict(n_train=300, epocas=1)
# lr 5e-5 con warmup: tasa estándar de ajuste de GPT-2 (el guía usa 2e-5 por 10 épocas
# sobre ~2.400 chistes; aquí hay 32.000 reseñas). Ver docs/DECISIONS.md §D-407.
CFG_GPT.update(lr=5e-5, warmup_frac=0.05, weight_decay=0.01, fp16=HAY_GPU, seed=SEED)

def fijar_semilla_hf(sem: int = SEED):
    # fijar_semilla (heredada) + la semilla interna de transformers
    fijar_semilla(sem); set_seed(sem)

DIR_RES = Path("../results") if Path("../results").exists() else Path("results")
(DIR_RES / "figures").mkdir(parents=True, exist_ok=True); (DIR_RES / "metrics").mkdir(parents=True, exist_ok=True)

print("transformers:", transformers.__version__, "| checkpoint:", CHECKPOINT)
for k, v in {**CFG_GPT, **{f"est.{k}": v for k, v in CFG_EST.items()}}.items():
    print(f"  {k:14s}: {v}")

### 4.5 El tokenizador BPE de GPT-2 frente a este corpus

GPT-2 usa BPE a nivel de byte: cualquier cadena se puede representar, sin `[UNK]`. El costo de
una palabra desconocida es la **longitud**: se parte en más piezas. El vocabulario de
`mrm8488/spanish-gpt2` se aprendió sobre español, así que esperamos una fertilidad mucho menor
que la de un GPT-2 inglés sobre el mismo texto.

Añadimos un token de relleno propio, `<|pad|>`. El guía usa el mismo token para relleno y para
fin de texto, y eso tiene una consecuencia seria: el *collator* excluye el relleno de la
pérdida, así que **el fin de texto nunca se aprende** y el modelo no sabe cuándo parar
(`docs/DECISIONS.md` §D-403).

In [ ]:
# =========================================================================
# 4.5 · Tokenizador, fertilidad y MAX_LEN_GPT
# =========================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM

def cargar_tokenizer(ckpt):
    tok = AutoTokenizer.from_pretrained(ckpt)
    if tok.pad_token is None or tok.pad_token_id == tok.eos_token_id:
        tok.add_special_tokens({"pad_token": "<|pad|>"})
    return tok

tokenizer = cargar_tokenizer(CHECKPOINT)
EOS = tokenizer.eos_token

y_va_A = sub.iloc[SPLIT_A["val"]]["Polarity"].values
txt_tr = sub.iloc[SPLIT_A["train"]]["texto"].tolist()
txt_va = sub.iloc[SPLIT_A["val"]]["texto"].tolist()
txt_te = sub.iloc[SPLIT_A["test"]]["texto"].tolist()

def fertilidad(tok, textos):
    ids = tok(list(textos), add_special_tokens=False)["input_ids"]
    return sum(map(len, ids)) / sum(len(t.split()) for t in textos)

print(f"Vocabulario: {len(tokenizer):,} (incluye <|pad|>) | EOS = {EOS!r} | pad = {tokenizer.pad_token!r}")
print(f"Fertilidad {CHECKPOINT}: {fertilidad(tokenizer, txt_tr[:3000]):.2f} tokens/palabra")
print(f"Fertilidad gpt2 (inglés), referencia: {fertilidad(AutoTokenizer.from_pretrained('gpt2'), txt_tr[:3000]):.2f} tokens/palabra")
frase = "El cenote de Tulum estaba padrísimo, pero el hostal chafa."
print(f"\n{frase}\n -> {tokenizer.tokenize(frase)}")

Los tokens de la salida aparecen con caracteres raros (`Ġ`, `Ã`) porque BPE byte-level muestra
los bytes crudos: `Ġ` es un espacio y las tildes ocupan dos bytes. Por eso el `generate` del
guía, que decodifica **token a token**, puede romper tildes y eñes: medio carácter no se puede
decodificar solo (lo corregimos en §8).

### 4.6 El formato de entrenamiento: prefijo de control

Cada reseña se convierte en una secuencia

`<EOS> Reseña de hotel con 1 estrella:⏎ <texto de la reseña> <EOS>`

El `<EOS>` inicial hace de inicio de secuencia (GPT-2 no tiene uno propio) y el final enseña a
terminar. El **prefijo en texto plano**, en lugar de tokens especiales, tiene dos ventajas: el
modelo base lo puede leer sin ajuste (así §5 prueba un *prompting* justo) y LoRA (§9) no
necesita aprender embeddings nuevos (`docs/DECISIONS.md` §D-404).

La perplejidad se calcula **solo sobre los tokens de la reseña**, nunca sobre el prefijo: así el
modelo sin condición (§6) y el condicional (§7) se miden sobre exactamente los mismos tokens.

In [ ]:
# =========================================================================
# 4.6 · Prefijo de control, MAX_LEN_GPT y conjuntos
# =========================================================================
TIPO_ES = {"Hotel": "hotel", "Restaurant": "restaurante", "Attractive": "atracción turística"}

def prefijo(estrellas, tipo):
    return f"Reseña de {TIPO_ES[tipo]} con {estrellas} estrella{'s' if estrellas != 1 else ''}:\n"

tipo_tr = sub.iloc[SPLIT_A["train"]]["Type"].values
tipo_va = sub.iloc[SPLIT_A["val"]]["Type"].values
tipo_te = sub.iloc[SPLIT_A["test"]]["Type"].values
pref_tr = [prefijo(e, t) for e, t in zip(y_tr_A, tipo_tr)]
pref_va = [prefijo(e, t) for e, t in zip(y_va_A, tipo_va)]
pref_te = [prefijo(e, t) for e, t in zip(y_te_A, tipo_te)]

long_tr = np.array([len(i) for i in tokenizer([p + t for p, t in zip(pref_tr, txt_tr)])["input_ids"]]) + 2
p95 = int(np.percentile(long_tr, 95))
MAX_LEN_GPT = int(min(CFG_GPT["max_len_tope"], int(np.ceil(p95 / 8) * 8)))
print(f"Longitud con prefijo y EOS: mediana {np.median(long_tr):.0f} | P95 {p95} -> MAX_LEN_GPT = {MAX_LEN_GPT}")
print(f"Reseñas de train truncadas: {(long_tr > MAX_LEN_GPT).mean()*100:.1f}%")
print("\nEjemplo de secuencia de entrenamiento:\n" + EOS + pref_tr[0] + txt_tr[0][:200] + " ..." + EOS)

Con esos textos armamos los conjuntos para el `Trainer`. En CPU se limita el número de reseñas
de entrenamiento; la validación usa un subconjunto fijo para que evaluar cada época no cueste
más que entrenarla.

In [ ]:
# --- Conjuntos tokenizados (con y sin prefijo) ---
from datasets import Dataset

def idx_train(n=None):
    """Índices estratificados por estrellas del train de Split A (todos si n es None)."""
    n_tot = len(txt_tr)
    if n is None or n >= n_tot:
        return np.arange(n_tot)
    sel, _ = train_test_split(np.arange(n_tot), train_size=n, random_state=SEED, stratify=y_tr_A)
    return np.sort(sel)

def a_dataset_lm(tok, textos, prefijos=None):
    prefijos = prefijos or [""] * len(textos)
    secs = [tok.eos_token + p + t + tok.eos_token for p, t in zip(prefijos, textos)]
    ds_ = Dataset.from_dict({"text": secs})
    return ds_.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN_GPT),
                   batched=True, remove_columns=["text"])

SEL_TR = idx_train(CFG_GPT["n_train_max"])
rng_val = np.random.RandomState(SEED)
SEL_VA = np.sort(rng_val.choice(len(txt_va), size=min(1_000, len(txt_va)), replace=False))
# Subconjunto fijo de test para perplejidad y clasificación generativa
SEL_TE = np.sort(np.random.RandomState(SEED).choice(len(txt_te), size=min(CFG_GPT["n_eval"], len(txt_te)), replace=False))

print(f"Train: {len(SEL_TR):,} | val (pérdida por época): {len(SEL_VA):,} | test (perplejidad): {len(SEL_TE):,}")

### 4.7 Los jueces: ¿se lee como lo que se pidió?

Para medir el control hace falta un lector automático de polaridad. Usamos exactamente el
modelo 1 del Miniproyecto 1 (TF-IDF + LogReg), entrenado solo con reseñas **reales** de train, y
uno igual para el tipo de establecimiento.

Un juez imperfecto no puede certificar más control del que él mismo percibe. Por eso lo
calibramos primero sobre reseñas reales de test: si sobre reseñas reales de 3★ acierta solo el
40 %, no es razonable exigir más a las reseñas generadas a 3★ (`docs/DECISIONS.md` §D-406).

In [ ]:
# =========================================================================
# 4.7 · Jueces TF-IDF + LogReg (modelo 1 de MP1) para estrellas y tipo
# =========================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def tfidf_mp1():
    # Mismo modelo 1 del Miniproyecto 1 (§5), con el tokenizar() heredado
    return Pipeline([
        ("tfidf", TfidfVectorizer(tokenizer=tokenizar, token_pattern=None, ngram_range=(1, 2),
                                  min_df=3, sublinear_tf=True, max_features=60000)),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0)),
    ])

juez_estrellas = tfidf_mp1().fit(txt_tr, y_tr_A)
juez_tipo = tfidf_mp1().fit(txt_tr, tipo_tr)

pred_real = juez_estrellas.predict(txt_te)
ACIERTO_REAL = {e: (pred_real[y_te_A == e] == e).mean() for e in ESTRELLAS}
acc_tipo_real = (juez_tipo.predict(txt_te) == tipo_te).mean()
print(f"Juez de estrellas sobre reseñas reales de test: macro-F1 "
      f"{f1_score(y_te_A, pred_real, average='macro'):.4f} (MP1: 0.5235)")
print("Acierto por estrella en reales:", {f"{e}★": round(v, 3) for e, v in ACIERTO_REAL.items()})
print(f"Juez de tipo sobre reseñas reales de test: accuracy {acc_tipo_real:.4f}")

### 4.8 Métricas de generación

- **Perplejidad**: exponencial de la pérdida media por token sobre reseñas reales no vistas.
  Si vale 20, el modelo duda en promedio como si eligiera entre 20 tokens equiprobables.
- **distinct-n**: proporción de n-gramas distintos entre todas las muestras. Baja si el modelo
  repite las mismas frases en muestras diferentes.
- **rep-4**: proporción de 4-gramas que se repiten dentro de un mismo texto; mide los bucles
  («muy bonito muy bonito muy bonito»).
- **Control**: acierto del juez sobre la estrella (o el tipo) pedida.

In [ ]:
# =========================================================================
# 4.8 · Perplejidad y métricas de texto generado
# =========================================================================
import torch.nn.functional as F

@torch.no_grad()
def nll_por_texto(modelo, tok, textos, prefijos=None, batch=None):
    """Suma de -log p y número de tokens de CADA reseña, sin contar el prefijo.
    Secuencia: EOS + prefijo + texto + EOS (el EOS final sí cuenta: saber terminar)."""
    modelo.eval(); disp = next(modelo.parameters()).device
    prefijos = prefijos or [""] * len(textos)
    tok.padding_side = "right"
    sumas, cuentas = [], []
    for i in range(0, len(textos), batch or CFG_GPT["batch_eval"]):
        ps, ts = prefijos[i:i + (batch or CFG_GPT["batch_eval"])], textos[i:i + (batch or CFG_GPT["batch_eval"])]
        enc = tok([tok.eos_token + p + t + tok.eos_token for p, t in zip(ps, ts)], truncation=True,
                  max_length=MAX_LEN_GPT, padding=True, return_tensors="pt").to(disp)
        n_pref = torch.tensor([len(tok(tok.eos_token + p)["input_ids"]) for p in ps], device=disp)
        with torch.autocast(disp.type, enabled=HAY_GPU):
            logits = modelo(**enc).logits[:, :-1].float()
        obj = enc["input_ids"][:, 1:]
        mask = enc["attention_mask"][:, 1:].bool()
        pos = torch.arange(obj.shape[1], device=disp).unsqueeze(0)
        mask &= pos >= (n_pref.unsqueeze(1) - 1)       # fuera el prefijo
        nll = F.cross_entropy(logits.transpose(1, 2), obj, reduction="none") * mask
        sumas.append(nll.sum(1).cpu()); cuentas.append(mask.sum(1).cpu())
    return torch.cat(sumas).numpy(), torch.cat(cuentas).numpy()

def perplejidad(modelo, tok, textos, prefijos=None):
    s, n = nll_por_texto(modelo, tok, textos, prefijos)
    return float(np.exp(s.sum() / n.sum()))

def ngramas(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def distinct_n(textos, n):
    todos = [g for t in textos for g in ngramas(tokenizar(t), n)]
    return len(set(todos)) / max(len(todos), 1)

def rep_4(textos):
    vals = []
    for t in textos:
        g = ngramas(tokenizar(t), 4)
        if g:
            vals.append(1 - len(set(g)) / len(g))
    return float(np.mean(vals)) if vals else 0.0

def control(textos, estrellas_pedidas, tipos_pedidos=None):
    """Acierto del juez sobre lo pedido. Textos vacíos cuentan como fallo."""
    textos = [t if t.strip() else "." for t in textos]
    pe = juez_estrellas.predict(textos)
    r = {"control_estrellas": float((pe == np.asarray(estrellas_pedidas)).mean()),
         "mae_estrellas": float(np.abs(pe - np.asarray(estrellas_pedidas)).mean())}
    if tipos_pedidos is not None:
        r["control_tipo"] = float((juez_tipo.predict(textos) == np.asarray(tipos_pedidos)).mean())
    return r, pe

print("Métricas listas: perplejidad, distinct-n, rep-4, control.")

In [ ]:
# --- Generación por lotes (usada en §5 a §14) ---
@torch.no_grad()
def generar(modelo, tok, prompts, semilla=SEED, max_new=None, **kw_decodificacion):
    """Genera una continuación por prompt. Devuelve solo el texto nuevo, sin el prompt."""
    modelo.eval(); disp = next(modelo.parameters()).device
    tok.padding_side = "left"                     # GPT-2 continúa por la derecha
    salidas = []
    torch.manual_seed(semilla)
    for i in range(0, len(prompts), CFG_GPT["batch_gen"]):
        enc = tok([tok.eos_token + p for p in prompts[i:i + CFG_GPT["batch_gen"]]],
                  return_tensors="pt", padding=True).to(disp)
        out = modelo.generate(**enc, max_new_tokens=max_new or CFG_GPT["max_new"],
                              pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
                              **kw_decodificacion)
        salidas += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return [s.strip() for s in salidas]

def prompts_control(n_por_estrella, tipos=None, semilla=SEED):
    """n prompts por estrella; el tipo se sortea con la distribución de train."""
    rng = np.random.RandomState(semilla)
    ests, tips = [], []
    for e in ESTRELLAS:
        ests += [e] * n_por_estrella
        tips += list(rng.choice(tipo_tr, size=n_por_estrella)) if tipos is None else [tipos] * n_por_estrella
    return [prefijo(e, t) for e, t in zip(ests, tips)], np.array(ests), np.array(tips)

---

## 5. El modelo base, sin ajustar

Empezamos como el guía: ¿qué distribución asigna GPT-2 al siguiente token? Pero con un inicio de
nuestro dominio. Es la unidad mínima de todo lo que sigue: generar es repetir este paso.

In [ ]:
# =========================================================================
# 5 · Distribución del siguiente token en el modelo base
# =========================================================================
fijar_semilla_hf()
base = AutoModelForCausalLM.from_pretrained(CHECKPOINT)
base.resize_token_embeddings(len(tokenizer))
base = base.to(DISPOSITIVO).eval()
N_PARAMS_GPT = sum(p.numel() for p in base.parameters())
print(f"{CHECKPOINT}: {N_PARAMS_GPT/1e6:.0f} M parámetros, "
      f"{base.config.n_layer} bloques decodificadores, contexto de {base.config.n_positions} tokens")

inicio = "El hotel estaba"
with torch.no_grad():
    ids = tokenizer(inicio, return_tensors="pt")["input_ids"].to(DISPOSITIVO)
    probs = torch.softmax(base(input_ids=ids).logits[0, -1].float(), -1)
top = torch.topk(probs, 10)
pd.DataFrame({"siguiente token": [repr(tokenizer.decode(i)) for i in top.indices.tolist()],
              "probabilidad (%)": (top.values.cpu().numpy() * 100).round(2)})

La perplejidad del modelo base sobre reseñas reales mide cuánto se parece el corpus de
preentrenamiento a nuestro dominio. La comparamos para **tres GPT-2 en español del mismo
tamaño** (12 bloques, 768 dimensiones) preentrenados sobre corpus distintos: el del guía,
uno sobre libros (DeepESP) y uno sobre Wikipedia (datificate). Cada uno con su propio
tokenizador; como tokenizan distinto, la perplejidad por token no es del todo comparable entre
ellos, así que reportamos también la **perplejidad por palabra**, que sí lo es.

In [ ]:
# =========================================================================
# 5 · Perplejidad en test de tres checkpoints base
# =========================================================================
txt_eval = [txt_te[i] for i in SEL_TE]
n_palabras = sum(len(t.split()) + 1 for t in txt_eval)      # +1: el EOS final
PPL = []
for nombre, ckpt in {"mrm8488 (guía)": CHECKPOINT, **OTROS_CHECKPOINTS}.items():
    if ckpt == CHECKPOINT:
        m, tk = base, tokenizer
    else:
        tk = cargar_tokenizer(ckpt)
        m = AutoModelForCausalLM.from_pretrained(ckpt); m.resize_token_embeddings(len(tk)); m.to(DISPOSITIVO)
    s, n = nll_por_texto(m, tk, txt_eval)
    PPL.append({"modelo": f"base · {nombre}", "perplejidad (token)": float(np.exp(s.sum() / n.sum())),
                "perplejidad (palabra)": float(np.exp(s.sum() / n_palabras)),
                "tokens/palabra": n.sum() / n_palabras})
    if ckpt != CHECKPOINT:
        del m; torch.cuda.empty_cache()
pd.DataFrame(PPL).round(2)

¿Obedece el modelo base al prefijo sin haber sido ajustado? Le pedimos reseñas de 1★ y de 5★
con el mismo prefijo que usaremos después, y el juez las lee.

In [ ]:
# --- Prompting sin ajuste: ¿el modelo base respeta la estrella pedida? ---
fijar_semilla_hf()
pr, est_ped, tip_ped = prompts_control(CFG_GPT["n_dec"])
gen_base = generar(base, tokenizer, pr, do_sample=True, top_p=0.92, top_k=0)
CTRL_BASE, _ = control(gen_base, est_ped, tip_ped)
print("Control del modelo base (prompting):", {k: round(v, 3) for k, v in CTRL_BASE.items()})
for e in (1, 5):
    i = int(np.where(est_ped == e)[0][0])
    print(f"\n[{pr[i].strip()}]\n{gen_base[i][:300]}")

<!-- LEER: qué checkpoint conoce mejor el dominio y por qué (corpus de preentrenamiento); si el base obedece el prefijo. -->

---

## 6. Técnica 1: *fine-tuning* sin condición

Lo mismo que hace el guía con los chistes: seguir entrenando el modelo en su tarea original
(predecir el siguiente token) sobre nuestras reseñas, sin ninguna indicación de estrellas. Es el
punto de comparación de la técnica 2: aprenderá el estilo, pero no tiene forma de saber qué
polaridad escribir, así que debería imitar la distribución del corpus (dos de cada tres, 5★).

Cambios frente al guía: relleno propio (§4.5), semilla, mejor modelo por pérdida de
validación, tasa 5e-5 con *warmup* y sin *checkpoints* en disco (`docs/DECISIONS.md` §D-407).

In [ ]:
# =========================================================================
# 6 · Utilidades de fine-tuning causal (reutilizadas en §7 y §9)
# =========================================================================
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def n_entrenables(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def afinar_lm(modelo, tok, train, val, epocas, lr=None, batch=None):
    """Ajuste causal reproducible. El mejor modelo (menor pérdida de validación) queda en
    memoria; los checkpoints del directorio temporal se borran. Devuelve (trainer, segundos)."""
    fijar_semilla_hf()
    salida = tempfile.mkdtemp(prefix="mp4_")
    batch = batch or CFG_GPT["batch"]
    args = TrainingArguments(
        output_dir=salida, num_train_epochs=epocas, learning_rate=lr or CFG_GPT["lr"],
        per_device_train_batch_size=batch, per_device_eval_batch_size=CFG_GPT["batch_eval"],
        warmup_steps=int(np.ceil(len(train) / batch) * epocas * CFG_GPT["warmup_frac"]),
        weight_decay=CFG_GPT["weight_decay"], fp16=CFG_GPT["fp16"],
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_strategy="steps", logging_steps=max(1, len(train) // batch // 8),
        report_to="none", seed=SEED, data_seed=SEED, disable_tqdm=True)
    tok.padding_side = "right"
    tr = Trainer(model=modelo, args=args, train_dataset=train, eval_dataset=val,
                 data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
    t0 = time.time(); tr.train(); seg = time.time() - t0
    shutil.rmtree(salida, ignore_errors=True)
    return tr, seg

def curvas(trainer, titulo, archivo):
    h = pd.DataFrame(trainer.state.log_history)
    ht, hv = h.dropna(subset=["loss"]), h.dropna(subset=["eval_loss"])
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(ht["epoch"], np.exp(ht["loss"]), "-", alpha=0.7, label="train")
    ax.plot(hv["epoch"], np.exp(hv["eval_loss"]), "o-", label="validación")
    ax.set_yscale("log"); ax.set_xlabel("época"); ax.set_ylabel("perplejidad (log)")
    ax.set_title(titulo); ax.legend(); plt.tight_layout()
    plt.savefig(DIR_RES / "figures" / archivo, bbox_inches="tight"); plt.show()

In [ ]:
# =========================================================================
# 6 · Técnica 1: fine-tuning sin condición
# =========================================================================
ds_libre_tr = a_dataset_lm(tokenizer, [txt_tr[i] for i in SEL_TR])
ds_libre_va = a_dataset_lm(tokenizer, [txt_va[i] for i in SEL_VA])

fijar_semilla_hf()
m_libre = AutoModelForCausalLM.from_pretrained(CHECKPOINT)
m_libre.resize_token_embeddings(len(tokenizer))
tr_libre, seg_libre = afinar_lm(m_libre, tokenizer, ds_libre_tr, ds_libre_va, CFG_GPT["epocas"])
m_libre = tr_libre.model.eval()

ppl_libre = perplejidad(m_libre, tokenizer, txt_eval)
PPL.append({"modelo": "FT sin condición", "perplejidad (token)": ppl_libre})
print(f"Perplejidad en test: base {PPL[0]['perplejidad (token)']:.1f} -> FT sin condición {ppl_libre:.1f}"
      f" | entrenamiento: {seg_libre/60:.1f} min")
curvas(tr_libre, "Técnica 1: fine-tuning sin condición", "curvas_ft_libre.png")

Sin prefijo, el modelo escribe lo que el corpus le enseñó a escribir con más frecuencia. El
juez nos dice qué distribución de estrellas produce por sí solo.

In [ ]:
# --- ¿Qué polaridad escribe el modelo sin condición? ---
fijar_semilla_hf()
gen_libre = generar(m_libre, tokenizer, [""] * (5 * CFG_GPT["n_dec"]), do_sample=True, top_p=0.92, top_k=0)
dist_gen = pd.Series(juez_estrellas.predict([g or "." for g in gen_libre])).value_counts(normalize=True).reindex(ESTRELLAS, fill_value=0)
dist_real = pd.Series(y_tr_A).value_counts(normalize=True).reindex(ESTRELLAS)
display(pd.DataFrame({"generadas (según el juez) %": dist_gen * 100, "reales de train %": dist_real * 100}).round(1).T)
for g in gen_libre[:3]:
    print("•", g[:250])

<!-- LEER: caída de perplejidad; ¿sobreajuste en las curvas?; la distribución de estrellas generada imita el desbalance. -->

---

## 7. Técnica 2: *fine-tuning* condicional

Mismo modelo, mismos datos, mismos hiperparámetros. La única diferencia es el prefijo
`Reseña de <tipo> con <n> estrella(s):`. El modelo aprende la distribución del texto **dado** el
prefijo, y al generar podemos pedirle la polaridad y el tipo que queramos, incluidas las clases
que casi no vio (1★ y 2★ son el 5 % del corpus).

In [ ]:
# =========================================================================
# 7 · Técnica 2: fine-tuning condicional
# =========================================================================
ds_cond_tr = a_dataset_lm(tokenizer, [txt_tr[i] for i in SEL_TR], [pref_tr[i] for i in SEL_TR])
ds_cond_va = a_dataset_lm(tokenizer, [txt_va[i] for i in SEL_VA], [pref_va[i] for i in SEL_VA])

fijar_semilla_hf()
m_cond = AutoModelForCausalLM.from_pretrained(CHECKPOINT)
m_cond.resize_token_embeddings(len(tokenizer))
tr_cond, seg_cond = afinar_lm(m_cond, tokenizer, ds_cond_tr, ds_cond_va, CFG_GPT["epocas"])
m_cond = tr_cond.model.eval()

pref_eval = [pref_te[i] for i in SEL_TE]
ppl_cond = perplejidad(m_cond, tokenizer, txt_eval, pref_eval)
PPL.append({"modelo": "FT condicional (prefijo correcto)", "perplejidad (token)": ppl_cond})
print(f"Perplejidad en test (solo tokens de la reseña): sin condición {ppl_libre:.2f} | "
      f"condicional {ppl_cond:.2f} | entrenamiento: {seg_cond/60:.1f} min")
curvas(tr_cond, "Técnica 2: fine-tuning condicional", "curvas_ft_condicional.png")

tabla_ppl = pd.DataFrame(PPL)
tabla_ppl.to_csv(DIR_RES / "metrics" / "perplejidad.csv", index=False)
tabla_ppl.round(2)

La prueba central del control: pedimos `n` reseñas por estrella (con tipo sorteado) y cruzamos
la estrella pedida con la que lee el juez. La columna de referencia es el acierto del mismo juez
sobre reseñas **reales** de esa estrella.

In [ ]:
# --- Control: estrella pedida vs. estrella que lee el juez ---
fijar_semilla_hf()
pr, est_ped, tip_ped = prompts_control(CFG_GPT["n_dec"])
gen_cond = generar(m_cond, tokenizer, pr, do_sample=True, top_p=0.92, top_k=0)
CTRL_COND, pe = control(gen_cond, est_ped, tip_ped)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
cm = confusion_matrix(est_ped, pe, labels=ESTRELLAS)
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=cm, fmt="d", cmap="Blues", cbar=False, vmin=0, vmax=1,
            xticklabels=ESTRELLAS, yticklabels=ESTRELLAS, ax=axes[0])
axes[0].set_xlabel("estrella que lee el juez"); axes[0].set_ylabel("estrella pedida")
axes[0].set_title("Generadas: pedida vs. leída")
acc_gen = [(pe[est_ped == e] == e).mean() for e in ESTRELLAS]
x = np.arange(5)
axes[1].bar(x - 0.2, [ACIERTO_REAL[e] for e in ESTRELLAS], 0.4, color="#999999", label="reseñas reales")
axes[1].bar(x + 0.2, acc_gen, 0.4, color="#2166ac", label="reseñas generadas")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"{e}★" for e in ESTRELLAS]); axes[1].set_ylim(0, 1)
axes[1].set_ylabel("acierto del juez"); axes[1].legend(); axes[1].set_title("¿Se lee como lo pedido?")
plt.tight_layout(); plt.savefig(DIR_RES / "figures" / "control_estrellas.png", bbox_inches="tight"); plt.show()
print({k: round(v, 3) for k, v in CTRL_COND.items()},
      f"| referencia tipo en reales: {acc_tipo_real:.3f} | base sin ajuste: {CTRL_BASE['control_estrellas']:.3f}")

In [ ]:
# --- Una muestra por estrella, mismo tipo ---
fijar_semilla_hf()
for e in ESTRELLAS:
    g = generar(m_cond, tokenizer, [prefijo(e, "Hotel")], do_sample=True, top_p=0.92, top_k=0)[0]
    print(f"[{e}★ pedida · juez: {juez_estrellas.predict([g or '.'])[0]}★] {g[:280]}\n")

<!-- LEER: veredicto H1 (perplejidad) y H2 (control por estrella frente a la referencia real); control de tipo. -->

---

## 8. Estrategias de decodificación

El modelo da una distribución sobre el siguiente token; **decodificar** es decidir qué hacer
con ella. Primero revisamos la función `generate` que escribe el guía, y luego comparamos las
estrategias habituales sobre el modelo condicional.

### 8.1 La función `generate` del guía, corregida

La función del guía implementa una política ε-codiciosa: con probabilidad `eps` toma el token
más probable y si no, muestrea. Tiene tres detalles que corregimos (`docs/DECISIONS.md`
§D-405):

1. **Recalcula todo en cada paso.** Pasa la secuencia completa por el modelo para cada token
   nuevo: generar *n* tokens cuesta del orden de *n²*. Los modelos decodificadores guardan las
   claves y valores de la atención ya calculados (*KV cache*), y con ellos cada paso procesa
   un solo token.
2. **Decodifica token a token** y concatena. En BPE byte-level una tilde ocupa dos bytes que
   pueden caer en tokens distintos: decodificarlos por separado produce `�`.
3. **Semilla**: usa `np.random` sin fijar. Aquí, un `torch.Generator` con semilla.

Conservamos su semántica de `eps` (probabilidad de explotar, al revés de la convención de
aprendizaje por refuerzo, donde ε es la de explorar) y la documentamos.

In [ ]:
# =========================================================================
# 8.1 · generate() del guía, corregido: KV cache, decodificación final, semilla
# =========================================================================
@torch.no_grad()
def generar_eps(modelo, tok, inicio, max_new=50, eps=0.5, usar_cache=True, top_n=0, semilla=SEED):
    """Política ε-codiciosa del guía. eps = probabilidad de tomar el token más probable
    (eps=1: codiciosa; eps=0: muestreo puro). Devuelve (texto, iteraciones, segundos)."""
    modelo.eval(); disp = next(modelo.parameters()).device
    g = torch.Generator(device="cpu").manual_seed(semilla)
    ids = tok(tok.eos_token + inicio, return_tensors="pt")["input_ids"].to(disp)
    generados, iteraciones, pasado = [], [], None
    entrada = ids
    t0 = time.perf_counter()
    for _ in range(max_new):
        if usar_cache:
            out = modelo(input_ids=entrada, past_key_values=pasado, use_cache=True)
            pasado = out.past_key_values
        else:                                   # como el guía: toda la secuencia otra vez
            out = modelo(input_ids=torch.cat([ids] + generados, dim=1) if generados else ids)
        probs = torch.softmax(out.logits[0, -1].float(), -1)
        if torch.rand(1, generator=g).item() < eps:
            sig = probs.argmax().view(1, 1)
        else:
            sig = torch.multinomial(probs.cpu(), 1, generator=g).view(1, 1).to(disp)
        if top_n:
            top = torch.topk(probs, top_n)
            iteraciones.append({f"opción {k+1}": f"{tok.decode(i)!r} ({p:.3f})"
                                for k, (i, p) in enumerate(zip(top.indices.tolist(), top.values.tolist()))})
        generados.append(sig); entrada = sig
        if sig.item() == tok.eos_token_id:
            break
    seg = time.perf_counter() - t0
    texto = tok.decode(torch.cat(generados, dim=1)[0], skip_special_tokens=True)   # una sola vez
    return texto, pd.DataFrame(iteraciones), seg

inicio = prefijo(1, "Restaurant")
texto, it, _ = generar_eps(m_cond, tokenizer, inicio, max_new=12, eps=1.0, top_n=5)
print(inicio + texto); it

Dos comprobaciones. La primera es de corrección: con `eps = 1` la función debe producir
**exactamente** los mismos tokens que la decodificación codiciosa de Hugging Face. La segunda es
de costo: tiempo por reseña con y sin caché para longitudes crecientes.

In [ ]:
# --- Corrección: eps=1 coincide con la codiciosa de Hugging Face ---
propio, _, _ = generar_eps(m_cond, tokenizer, inicio, max_new=30, eps=1.0)
hf = generar(m_cond, tokenizer, [inicio], max_new=30, do_sample=False)[0]
print("propia :", propio.strip()[:120]); print("HF     :", hf[:120])
assert propio.strip() == hf.strip(), "La implementación propia no coincide con la codiciosa de HF"
print("OK: idénticas.")

# --- Costo: con y sin KV cache ---
tiempos = []
for n in [16, 32, 64, 128] if HAY_GPU else [8, 16, 32]:
    for cache in (True, False):
        _, _, s = generar_eps(m_cond, tokenizer, inicio, max_new=n, eps=1.0, usar_cache=cache)
        tiempos.append({"tokens": n, "caché": "con KV cache" if cache else "sin caché (guía)", "segundos": s})
tiempos = pd.DataFrame(tiempos)
fig, ax = plt.subplots(figsize=(7, 3.8))
for nombre, g in tiempos.groupby("caché"):
    ax.plot(g["tokens"], g["segundos"], "o-", label=nombre)
ax.set_xlabel("tokens generados"); ax.set_ylabel("segundos"); ax.legend()
ax.set_title("Costo de generar: el guía recalcula toda la secuencia en cada paso")
plt.tight_layout(); plt.savefig(DIR_RES / "figures" / "kv_cache.png", bbox_inches="tight"); plt.show()

### 8.2 Ocho estrategias comparadas

Sobre el modelo condicional, mismas peticiones (`n` por estrella), ocho formas de decodificar:

| Estrategia | Idea |
|---|---|
| Codiciosa | Siempre el token más probable. Determinista; propensa a bucles |
| Codiciosa + `no_repeat_ngram_size=3` | Prohíbe repetir trigramas |
| *Beam search* (4 haces) + `no_repeat` | Busca la secuencia más probable, no el token |
| Muestreo, T = 0,7 | Muestreo con la distribución afilada |
| Muestreo puro, T = 1 | La distribución tal cual |
| Top-k (k = 50) | Muestrea solo entre los 50 más probables |
| Top-p (p = 0,92) | Muestrea del núcleo que acumula el 92 % de la probabilidad |
| ε-codiciosa del guía (eps = 0,5) | Mitad codiciosa, mitad muestreo |

Métricas: diversidad (distinct-2), repetición (rep-4), control (juez), longitud y **fluidez**:
la perplejidad del texto generado bajo el **modelo base** sin ajustar, un lector de castellano
general que no participó en el ajuste. Valores altos indican texto raro o mal formado.

In [ ]:
# =========================================================================
# 8.2 · Comparación de estrategias de decodificación
# =========================================================================
ESTRATEGIAS = {
    "codiciosa": dict(do_sample=False),
    "codiciosa + no_repeat": dict(do_sample=False, no_repeat_ngram_size=3),
    "beam 4 + no_repeat": dict(do_sample=False, num_beams=4, no_repeat_ngram_size=3),
    "muestreo T=0.7": dict(do_sample=True, temperature=0.7, top_k=0),
    "muestreo puro T=1": dict(do_sample=True, temperature=1.0, top_k=0),
    "top-k 50": dict(do_sample=True, top_k=50),
    "top-p 0.92": dict(do_sample=True, top_p=0.92, top_k=0),
}
pr, est_ped, tip_ped = prompts_control(CFG_GPT["n_dec"])
MUESTRAS_DEC, DEC = {}, []
for nombre, kw in list(ESTRATEGIAS.items()) + [("ε-codiciosa (guía) eps=0.5", None)]:
    fijar_semilla_hf(); t0 = time.time()
    if kw is None:
        gens = [generar_eps(m_cond, tokenizer, p, max_new=CFG_GPT["max_new"], eps=0.5, semilla=SEED + k)[0].strip()
                for k, p in enumerate(pr)]
    else:
        gens = generar(m_cond, tokenizer, pr, **kw)
    seg = time.time() - t0
    ctrl, _ = control(gens, est_ped, tip_ped)
    no_vacios = [g for g in gens if g.strip()] or ["."]
    DEC.append({"estrategia": nombre, "distinct-2": distinct_n(gens, 2), "rep-4": rep_4(gens),
                "fluidez (PPL base)": perplejidad(base, tokenizer, no_vacios),
                "control estrellas": ctrl["control_estrellas"], "control tipo": ctrl["control_tipo"],
                "palabras": np.mean([len(g.split()) for g in gens]), "segundos": seg})
    MUESTRAS_DEC[nombre] = gens
DEC = pd.DataFrame(DEC)
DEC.to_csv(DIR_RES / "metrics" / "decodificacion.csv", index=False)
DEC.round(3)

In [ ]:
# --- Compromiso diversidad vs. control (tamaño = fluidez: más grande, más raro) ---
fig, ax = plt.subplots(figsize=(8.5, 5))
tam = 60 + 400 * (DEC["fluidez (PPL base)"] / DEC["fluidez (PPL base)"].max())
ax.scatter(DEC["distinct-2"], DEC["control estrellas"], s=tam, c=DEC["rep-4"], cmap="Reds", edgecolor="k")
for _, r in DEC.iterrows():
    ax.annotate(r["estrategia"], (r["distinct-2"], r["control estrellas"]), fontsize=8,
                xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("diversidad (distinct-2)"); ax.set_ylabel("control de estrellas (acierto del juez)")
ax.set_title("Decodificación: diversidad, control, repetición (color) y rareza (tamaño)")
plt.tight_layout(); plt.savefig(DIR_RES / "figures" / "decodificacion.png", bbox_inches="tight"); plt.show()

print("Misma petición (1★, restaurante) con tres estrategias:")
i = int(np.where(est_ped == 1)[0][0])
for n in ["codiciosa", "beam 4 + no_repeat", "top-p 0.92"]:
    print(f"\n[{n}] {MUESTRAS_DEC[n][i][:250]}")

<!-- LEER: qué estrategia elegimos para el resto (§10 a §14) y por qué; bucles de la codiciosa; beam vs muestreo. -->

---

## 9. Estudios: LoRA y otro checkpoint

> Con `CFG_EST`: 8.000 reseñas de train y una época. Sus filas se comparan entre sí, no con §7.

**LoRA** (*Low-Rank Adaptation*, Hu et al. 2021) congela GPT-2 y aprende una corrección de
rango bajo en la proyección de atención (`c_attn`). No se vio en clase; es como hoy se ajustan
los modelos grandes. Y repetimos el ajuste completo con **DeepESP** (preentrenado con libros)
para ver si el punto de partida importa una vez ajustado.

In [ ]:
# =========================================================================
# 9 · Ajuste completo vs. LoRA vs. otro checkpoint, con CFG_EST
# =========================================================================
from peft import LoraConfig, get_peft_model, TaskType

SEL_EST = idx_train(CFG_EST["n_train"])
EST = []

def correr_estudio(nombre, ckpt, lora=False, lr=None):
    tk = tokenizer if ckpt == CHECKPOINT else cargar_tokenizer(ckpt)
    tr_ds = a_dataset_lm(tk, [txt_tr[i] for i in SEL_EST], [pref_tr[i] for i in SEL_EST])
    va_ds = a_dataset_lm(tk, [txt_va[i] for i in SEL_VA], [pref_va[i] for i in SEL_VA])
    fijar_semilla_hf()
    m = AutoModelForCausalLM.from_pretrained(ckpt); m.resize_token_embeddings(len(tk))
    if lora:
        m = get_peft_model(m, LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
                                         lora_dropout=0.05, target_modules=["c_attn"], fan_in_fan_out=True))
    entrenables = n_entrenables(m); total = sum(p.numel() for p in m.parameters())
    tr, seg = afinar_lm(m, tk, tr_ds, va_ds, CFG_EST["epocas"], lr=lr)
    m = tr.model.eval()
    s, n = nll_por_texto(m, tk, txt_eval, pref_eval)
    fijar_semilla_hf()
    pr_, e_, t_ = prompts_control(CFG_GPT["n_dec"])
    ctrl, _ = control(generar(m, tk, pr_, do_sample=True, top_p=0.92, top_k=0), e_, t_)
    EST.append({"variante": nombre, "perplejidad (token)": float(np.exp(s.sum() / n.sum())),
                "perplejidad (palabra)": float(np.exp(s.sum() / n_palabras)),
                **ctrl, "% entrenable": entrenables / total * 100, "segundos": seg})
    del m, tr; torch.cuda.empty_cache()

correr_estudio("mrm8488 · ajuste completo", CHECKPOINT)
correr_estudio("mrm8488 · LoRA r=8", CHECKPOINT, lora=True, lr=5e-4)   # LoRA necesita tasas mayores
correr_estudio("DeepESP · ajuste completo", OTROS_CHECKPOINTS["DeepESP (libros)"])
pd.DataFrame(EST).round(3)

<!-- LEER: cuánto pierde LoRA con <1% entrenable; si el checkpoint de partida importa tras ajustar. -->

---

## 10. El generador como clasificador

Un modelo condicional no solo genera: también **puntúa**. Para una reseña real calculamos su
perplejidad bajo cada uno de los cinco prefijos (1★ a 5★, con su tipo verdadero) y predecimos la
estrella que mejor la explica:

$$\hat{e} = \arg\max_e \; p(\text{reseña} \mid \text{prefijo}_e)$$

Es clasificación **generativa** (regla de Bayes con prior uniforme), lo opuesto a los
clasificadores discriminativos de MP1 a MP3. Con prior uniforme no favorece a 5★, así que es
esperable que gane en las clases minoritarias y pierda *accuracy*. La evaluamos con la misma
`evaluar` heredada, sobre el subconjunto fijo de test.

In [ ]:
# =========================================================================
# 10 · Clasificación generativa con el modelo condicional
# =========================================================================
tipos_eval = [tipo_te[i] for i in SEL_TE]
y_eval = y_te_A[SEL_TE]
nll = np.stack([nll_por_texto(m_cond, tokenizer, txt_eval, [prefijo(e, t) for t in tipos_eval])[0]
                for e in ESTRELLAS], axis=1)                     # (n, 5), -log p por estrella
pred_gen = np.array(ESTRELLAS)[nll.argmin(1)]

# Con el prior de train (Bayes completo) en lugar de uniforme
log_prior = np.log(pd.Series(y_tr_A).value_counts(normalize=True).reindex(ESTRELLAS).values)
pred_gen_prior = np.array(ESTRELLAS)[(-nll + log_prior).argmax(1)]

evaluar(y_eval, pred_gen, "GPT-2 condicional como clasificador (prior uniforme)", n_params=N_PARAMS_GPT)
evaluar(y_eval, pred_gen_prior, "GPT-2 condicional como clasificador (prior de train)", n_params=N_PARAMS_GPT)
evaluar(y_eval, juez_estrellas.predict(txt_eval), "TF-IDF + LogReg (MP1) en el mismo subconjunto", registrar=False)

<!-- LEER: ¿compite con TF-IDF? efecto del prior en accuracy vs macro-F1; qué dice de lo que aprendió el modelo. -->

---

## 11. ¿Memoriza?

Un generador que copia reseñas del entrenamiento parecería excelente en todas las métricas
anteriores. Medimos dos cosas sobre reseñas generadas: la fracción de sus 8-gramas de palabras
que aparece literalmente en el train, y su similitud TF-IDF máxima con cualquier reseña de
train. La referencia justa son **reseñas reales de test**, que tampoco se vieron: dos reseñas
reales sobre el mismo hotel también se parecen.

In [ ]:
# =========================================================================
# 11 · Copia literal (8-gramas) y similitud con el train
# =========================================================================
from sklearn.metrics.pairwise import linear_kernel

ocho_train = set()
for t in txt_tr:
    ocho_train.update(hash(g) for g in ngramas(tokenizar(t), 8))

def frac_copia(textos):
    vals = [np.mean([hash(g) in ocho_train for g in ngramas(tokenizar(t), 8)])
            for t in textos if len(tokenizar(t)) >= 8]
    return np.array(vals)

gen_mem = MUESTRAS_DEC["top-p 0.92"]
reales_mem = [txt_te[i] for i in SEL_TE[:len(gen_mem)]]
vec = juez_estrellas.named_steps["tfidf"]
X_tr = vec.transform(txt_tr)
sim_max = lambda textos: linear_kernel(vec.transform(textos), X_tr).max(1)

copia_gen, copia_real = frac_copia(gen_mem), frac_copia(reales_mem)
s_gen, s_real = sim_max(gen_mem), sim_max(reales_mem)
print(f"8-gramas presentes en train: generadas {copia_gen.mean()*100:.1f}% | reales de test {copia_real.mean()*100:.1f}%")
print(f"Generadas con ≥ 50% de 8-gramas copiados: {(copia_gen >= 0.5).mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist([copia_real, copia_gen], bins=20, label=["reales de test", "generadas"], color=["#999999", "#2166ac"])
axes[0].set_xlabel("fracción de 8-gramas presentes en train"); axes[0].legend()
axes[1].hist([s_real, s_gen], bins=20, label=["reales de test", "generadas"], color=["#999999", "#2166ac"])
axes[1].set_xlabel("similitud TF-IDF máxima con el train"); axes[1].legend()
plt.suptitle("¿Copia el generador?"); plt.tight_layout()
plt.savefig(DIR_RES / "figures" / "memorizacion.png", bbox_inches="tight"); plt.show()

j = int(np.argmax(s_gen))
print(f"\nGenerada más parecida a una de train (sim {s_gen[j]:.2f}):\n  {gen_mem[j][:250]}")
print(f"  train: {txt_tr[int(linear_kernel(vec.transform([gen_mem[j]]), X_tr).argmax())][:250]}")

<!-- LEER: ¿generadas más parecidas al train que las reales? ¿casos de copia? -->

---

## 12. Aumento de datos para el clasificador de MP1

Las tres entregas anteriores chocaron con lo mismo: 1★ y 2★ tienen ~840 reseñas de train cada
una, y ningún modelo pasó de F1 ≈ 0,3 en 2★. Si el generador escribe reseñas creíbles de esas
clases, deberían servir para entrenar.

Tres versiones del modelo 1 de MP1 (TF-IDF + LogReg), evaluadas en **el mismo test de MP1**:

1. Train real (debe reproducir el 0,5235 de MP1).
2. Train real + **sobremuestreo**: se duplican reseñas reales de 1★, 2★ y 3★ al azar, tantas
   como las sintéticas de la versión 3. Es el control: separa el efecto de «más ejemplos de la
   clase» del efecto de «ejemplos nuevos».
3. Train real + **reseñas sintéticas** de 1★, 2★ y 3★ del generador condicional (top-p).

Las sintéticas se añaden solo al train; ni validación ni test las ven.

In [ ]:
# =========================================================================
# 12 · Aumento de datos: real vs. sobremuestreo vs. sintéticas
# =========================================================================
CLASES_AUM = [1, 2, 3]
fijar_semilla_hf()
t0 = time.time()
pr_aug, est_aug, _ = prompts_control(CFG_GPT["n_aug"], semilla=SEED + 1)
msk = np.isin(est_aug, CLASES_AUM)
pr_aug, est_aug = [p for p, k in zip(pr_aug, msk) if k], est_aug[msk]
sint = generar(m_cond, tokenizer, pr_aug, do_sample=True, top_p=0.92, top_k=0)
ok = np.array([len(s.split()) >= 3 for s in sint])                # se descartan vacías o triviales
sint, est_aug = [s for s, k in zip(sint, ok) if k], est_aug[ok]
print(f"{len(sint):,} reseñas sintéticas generadas en {(time.time()-t0)/60:.1f} min "
      f"({(~ok).sum()} descartadas por vacías)")

rng = np.random.RandomState(SEED)
extra_idx = np.concatenate([rng.choice(np.where(y_tr_A == e)[0], size=(est_aug == e).sum(), replace=True)
                            for e in CLASES_AUM])
variantes = {
    "TF-IDF + LogReg · train real (MP1)": (txt_tr, y_tr_A),
    "TF-IDF + LogReg · + sobremuestreo 1-3★": (txt_tr + [txt_tr[i] for i in extra_idx],
                                              np.concatenate([y_tr_A, y_tr_A[extra_idx]])),
    "TF-IDF + LogReg · + sintéticas 1-3★": (txt_tr + sint, np.concatenate([y_tr_A, est_aug])),
}
AUM = {}
for nombre, (X, yy) in variantes.items():
    t0 = time.time(); m_ = tfidf_mp1().fit(X, yy); seg = time.time() - t0
    AUM[nombre] = evaluar(y_te_A, m_.predict(txt_te), nombre, segundos=seg)

cols = ["macro_f1", "accuracy", "mae", "qwk"] + [f"f1_{e}" for e in ESTRELLAS]
tabla_aum = pd.DataFrame(AUM).T[cols].astype(float)
tabla_aum.to_csv(DIR_RES / "metrics" / "aumento_datos.csv")
tabla_aum.round(4)

In [ ]:
# --- F1 por clase de las tres variantes ---
f1c = tabla_aum[[f"f1_{e}" for e in ESTRELLAS]]; f1c.columns = [f"{e}★" for e in ESTRELLAS]
ax = f1c.T.plot(kind="bar", figsize=(9, 4), color=["#999999", "#ef8a62", "#2166ac"], width=0.8)
ax.set_ylabel("F1 (test de MP1)"); ax.set_ylim(0, 1); plt.xticks(rotation=0)
ax.set_title("¿Ayudan las reseñas sintéticas a las clases escasas?"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(DIR_RES / "figures" / "aumento_datos.png", bbox_inches="tight"); plt.show()

<!-- LEER: veredicto H3; ¿sintéticas > sobremuestreo? ¿en qué clases? ¿algún costo en 4★/5★? -->

---

## 13. Demo

`generar_resena(estrellas, tipo, inicio)` escribe una reseña con lo pedido y muestra cómo la lee
el juez. Pares fijos, con texto y semilla fijos para que «Restart & Run All» los reproduzca:

| Par | Qué revela |
|---|---|
| Misma apertura «La comida» a 1★ y a 5★ | ¿La condición cambia la continuación de un mismo inicio? |
| 3★ hotel frente a 3★ restaurante | ¿Cambia el vocabulario según el tipo? |
| «No volvería» pedida a 5★ | ¿Qué gana: el inicio negativo o la condición positiva? |

In [ ]:
# =========================================================================
# 13 · Demo
# =========================================================================
def generar_resena(estrellas, tipo="Hotel", inicio="", semilla=SEED, **kw):
    kw = kw or dict(do_sample=True, top_p=0.92, top_k=0)
    fijar_semilla_hf(semilla)
    txt = inicio + " " + generar(m_cond, tokenizer, [prefijo(estrellas, tipo) + inicio], semilla=semilla, **kw)[0]
    txt = txt.strip()
    return txt, int(juez_estrellas.predict([txt])[0]), juez_tipo.predict([txt])[0]

PARES = [((1, "Restaurant", "La comida"), (5, "Restaurant", "La comida")),
         ((3, "Hotel", ""), (3, "Restaurant", "")),
         ((5, "Attractive", "No volvería"), (1, "Attractive", "No volvería"))]
for a, b in PARES:
    for e, t, ini in (a, b):
        txt, je, jt = generar_resena(e, t, ini)
        print(f"[pedida {e}★ · {TIPO_ES[t]} | juez: {je}★ · {TIPO_ES.get(jt, jt)}]\n  {txt[:300]}\n")
    print("-" * 90)

Para probar peticiones propias hay un widget. Es un extra: su salida no queda guardada y no
forma parte de los resultados.

In [ ]:
# --- Widget opcional (no afecta la reproducibilidad) ---
try:
    import ipywidgets as w
    e_w = w.IntSlider(value=1, min=1, max=5, description="estrellas")
    t_w = w.Dropdown(options=list(TIPO_ES), value="Hotel", description="tipo")
    i_w = w.Text(value="", description="inicio")
    b_w, o_w = w.Button(description="Generar"), w.Output()
    def _generar(_):
        with o_w:
            o_w.clear_output()
            txt, je, jt = generar_resena(e_w.value, t_w.value, i_w.value, semilla=int(time.time()))
            print(f"(el juez lee: {je}★, {TIPO_ES.get(jt, jt)})\n{txt}")
    b_w.on_click(_generar)
    display(w.VBox([w.HBox([e_w, t_w, i_w]), b_w, o_w]))
except Exception as e:
    print("Widget no disponible en este entorno:", e)

<!-- LEER: resultados de los tres pares. -->

---

## 14. Análisis cualitativo y modos de falla

Las métricas resumen; leer muestras enseña qué falla. Sobre las reseñas sintéticas de §12 (y
las de 4★ y 5★ de §8) contamos fallas detectables automáticamente:

- **Repetición**: rep-4 > 0,3.
- **Corta o truncada**: menos de 5 palabras, o termina sin puntuación al agotar `max_new`.
- **Polaridad cruzada**: el juez la lee a 2 o más estrellas de distancia de lo pedido.
- **Negación en 1★**: ¿reproducen el rasgo que el EDA §3.5 encontró en las reales?

In [ ]:
# =========================================================================
# 14 · Modos de falla y muestras por estrella
# =========================================================================
pr_d, e_d, _ = prompts_control(CFG_GPT["n_dec"])
g_d = MUESTRAS_DEC["top-p 0.92"]
todas = pd.DataFrame({"texto": sint + [g for g, e in zip(g_d, e_d) if e in (4, 5)],
                      "pedida": np.concatenate([est_aug, [e for e in e_d if e in (4, 5)]])})
todas["juez"] = juez_estrellas.predict([t or "." for t in todas["texto"]])
todas["repetición"] = [rep_4([t]) > 0.3 for t in todas["texto"]]
todas["corta/truncada"] = [len(t.split()) < 5 or t.rstrip()[-1:] not in ".!?)" for t in todas["texto"]]
todas["polaridad cruzada"] = (todas["juez"] - todas["pedida"]).abs() >= 2
fallas = todas.groupby("pedida")[["repetición", "corta/truncada", "polaridad cruzada"]].mean() * 100
display(fallas.round(1).rename(index=lambda e: f"{e}★"))

NEG = {"no", "ni", "nada", "nunca", "jamás", "tampoco"}
tasa_neg = lambda ts: np.mean([sum(w in NEG for w in tokenizar(t)) / max(len(tokenizar(t)), 1) * 100 for t in ts])
print(f"Negaciones por 100 palabras en 1★: generadas {tasa_neg(todas.loc[todas.pedida == 1, 'texto']):.2f} | "
      f"reales de train {tasa_neg([t for t, e in zip(txt_tr, y_tr_A) if e == 1][:2000]):.2f}")

pd.set_option("display.max_colwidth", 260)
display(pd.concat([g.sample(min(2, len(g)), random_state=SEED) for _, g in todas.groupby("pedida")])
        [["pedida", "juez", "texto"]])

<!-- LEER: fallas dominantes por estrella; ¿aparece la negación en 1★? -->

---

## 15. Conclusiones y limitaciones

<!-- REDACTAR tras ejecutar:
- Respuesta a la pregunta: ¿aprende la polaridad pedida? ¿sirve lo generado?
- Veredicto H1 (perplejidad), H2 (control por estrella frente a la referencia real), H3 (aumento).
- Qué estrategia de decodificación y qué técnica (completo vs LoRA) recomendaríamos.
- El generador como clasificador frente a MP1–MP3.
- Limitaciones: juez imperfecto (TF-IDF, macro-F1 0,52), una semilla, evaluación humana ausente,
  estudios con CFG_EST, subconjunto de test para perplejidad.
- Trabajo futuro: juez BETO (MP3), modelos mayores, filtrado de sintéticas por el juez,
  control por destino (Town).
-->